# 4test_UN — diffusivity options + dislocation diffusivity scaling

Standalone notebook derived from `3test_UN_guard_diagnostics.ipynb`.

Physics is intentionally kept the same except for explicit user-selectable diffusivity modes for Xe and U vacancies, plus separate dislocation-diffusivity multipliers.


In [127]:
# ============================================================
# USER SETTINGS — change this cell for quick manual tests
# ============================================================

# Output
OUTPUT_DIR = "4test_UN_results"
CASE_LABEL = "4test_UN"
SHOW_PLOTS = False          # False = save PNG and close figures; True = show inline too

# Numerics
DT_H = 12.0                 # Zullo fast-demo default
N_MODES = 22                # Zullo fast-demo default
# For final-quality plots try: DT_H = 1.0; N_MODES = 40

# Temperature / burnup grids
T_MIN = 900.0
T_MAX_SWELLING = 2000.0     # swelling plots usually compare to experimental range
T_MAX_DIAGNOSTIC = 2000.0   # radius/Nd/pressure/partition diagnostic range
T_STEP = 50.0
BURNUPS = [1.1, 1.3, 3.2]

# ------------------------------------------------------------
# Physics switches — visible on purpose
# ------------------------------------------------------------
# Zullo current/default demo: True, True, True
USE_PHI_GAS_RESOLUTION = False
USE_NUCLEATION_MASS_COUPLING = True
USE_BULK_DISLOCATION_CAPTURE = True

# ------------------------------------------------------------
# Bubble first-gas pressure initialization
# ------------------------------------------------------------
# New test requested:
# - do NOT put a gas dimer inside bubbles at t=0;
# - when a bubble population first receives gas, assign vacancies so that
#   p = FIRST_GAS_PRESSURE_FACTOR * p_eq at the end of that first gas step;
# - after that, leave normal vacancy absorption evolution active.
USE_FIRST_GAS_PRESSURE_SEED = True
FIRST_GAS_PRESSURE_FACTOR = 1.0 / 3.0
SEED_BULK_ON_FIRST_GAS = True
SEED_DISLOCATION_ON_FIRST_GAS = True

# Legacy option from the previous test. Keep False for the current test.
# If True, it seeds a dimer already at birth/t=0; this is NOT the current default.
USE_EQUILIBRIUM_DIMER_SEED = False
SEED_GAS_ATOMS_PER_BUBBLE = 2.0
SEED_PRESSURE_FACTOR = 1.0 / 3.0
SEED_INITIAL_DISLOCATION_BUBBLES = False
SEED_NEW_BULK_BUBBLES = False

# Numerical safety: avoid crashing if a runaway state makes psi=R/delta astronomically large.
# It does not fix the physics; it only lets plots/CSV finish and should be read with diagnostics.
USE_ZETA_OVERFLOW_GUARD = True

# ------------------------------------------------------------
# Physical constants / nominal values
# ------------------------------------------------------------
GRAIN_RADIUS = 6.0e-6
XE_YIELD = 0.24
GAMMA_B = 1.11
OMEGA_FG = 8.5e-29
LATTICE_PARAMETER = 4.889e-10

FISSION_RATE_NOMINAL = 5.77e19
F_N_NOMINAL = 1.0e-6
K_D_NOMINAL = 5.0e5       # 1.2e6  Zullo default. For Rizk 2025 Table 1-like use 5.0e5.
RHO_D_NOMINAL = 3.0e13

# ------------------------------------------------------------
# Diffusivity modes
# ------------------------------------------------------------
# Xe / fission-gas diffusivity mode.
# Keep the first option to reproduce the previous 3test_UN behavior.
# Options:
#   "rizk2025_table_D1D3"  : Rizk 2025 Table 2, Xe, with D2 neglected; Dg = D1 + D3.
#   "rizk2025_refit_plot" : refit of digitized Rizk 2025 D_gas curve; Dg = D1 + D2 + D3.
#   "rizk2023_table_raw"  : Rizk 2023 Table 3.1 raw Xe fit, Xei + {Xe:VU} + D3.
XE_DIFFUSIVITY_MODE = "rizk2025_refit_plot"

# U-vacancy diffusivity mode.
# Keep the first option to reproduce the previous 3test_UN behavior.
# Options:
#   "rizk2025_code_refit_old" : previous Zullo/v14-style refit used in the code.
#   "rizk2025_refit_A20_only" : digitized Rizk 2025 V_U curve, only A20 refitted.
#   "rizk2025_refit_full"     : digitized Rizk 2025 V_U curve, full D1+D2 refit.
#   "rizk2023_table_raw"      : Rizk 2023 Table 3.1 raw V_U fit.
VU_DIFFUSIVITY_MODE = "rizk2025_refit_full"

# Legacy constants kept for backward compatibility with the previous code-refit option.
A20_VU_REFIT = 4.6304523933553033e-29  # previous code/Zullo-v14 Fig. 4 refit
A20_VU_RIZK2025 = 1.32e-19             # Rizk 2025 Table 2 coefficient, not used by default
A20_VU_DEFAULT = A20_VU_REFIT
B21_VU = -0.62
B22_VU = -0.04

# ------------------------------------------------------------
# Manual case parameters and scale factors
# No Optuna candidate is hidden here: these are all editable.
# ------------------------------------------------------------
MANUAL_PARAMS = {
    "f_n": F_N_NOMINAL,
    "K_d": K_D_NOMINAL,
    "rho_d": RHO_D_NOMINAL,
    "fission_rate": FISSION_RATE_NOMINAL,

    "Dv_scale": 1.0,        # total U-vacancy diffusivity scale; Rizk 2023 calibrated model used x5
    "Dv_D1_scale": 1.0,
    "Dv_D2_scale": 1.0,
    "Dv_dislocation_scale": 10,  # dislocation vacancy diffusivity multiplier; Rizk 2023 effective value ≈ 10*D_VU

    "Dg_scale": 1.0,        # total Xe/gas diffusivity scale; Rizk 2023 calibrated model used x5
    "Dg_D1_scale": 1.0,
    "Dg_D3_scale": 1.0,
    "Dg_dislocation_scale": 10,  # Xe pipe/core diffusivity multiplier for dislocation line sink; Rizk 2023 effective value ≈ 10*D_Xe

    "b_scale": 1.0,
    "b_bulk_scale": 1.0,
    "b_dislocation_scale": 1.0,

    "gb_scale": 1.0,
    "gd_scale": 1.0,
    "gd_bubble_scale": 1.0,
    "gd_line_scale": 1.0,
    "gd_line_alpha": 1.0,

    "coalescence_d_scale": 1.0,
    "capture_scale": 1.0,
}

# Dislocation-density mode.
# "constant" uses MANUAL_PARAMS["rho_d"].
# "rhoSat_RayBlank" uses the v14-style saturating Ray/Blank shape with global scale RHO_SCALE.
# "rizk2023" uses Rizk 2023 Eq. (3.38)-style exponential dislocation density:
#     rho_d = rho0 * exp(c*F / max(Td, T0 - T)), capped at rho_cap.
#     IMPORTANT: F is FIMA fraction, e.g. 1.3% FIMA -> F = 0.013.
RHO_MODE = "constant"      # options: "constant", "rhoSat_RayBlank", "rizk2023"
RHO_SCALE = 1.0             # only used by rhoSat_RayBlank
RHO_FAB = 1e13              # only used by rhoSat_RayBlank

# Rizk 2023 empirical dislocation-density parameters, Table 3.3 value used for UN.
# Keep these unchanged to reproduce the published placeholder rho_d model.
RIZK2023_RHO0 = 1.0e12      # [m^-2]
RIZK2023_C = 49000.0        # [K / FIMA_fraction]
RIZK2023_T0 = 1900.0        # [K]
RIZK2023_TD = 150.0         # [K]
RIZK2023_RHO_CAP = 1.0e15   # [m^-2]


## Experimental data

In [128]:
# Experimental data digitized for comparison
"""Digitized experimental data used by the UN calibration pipeline."""

EXP_SWELLING_T = [
    {"figure": "Fig3a", "burnup": 1.1, "series": "100 kW/m", "T": 1127.0, "swelling": 0.68},
    {"figure": "Fig3a", "burnup": 1.1, "series": "100 kW/m", "T": 1228.0, "swelling": 0.59},
    {"figure": "Fig3a", "burnup": 1.1, "series": "100 kW/m", "T": 1312.0, "swelling": 0.43},
    {"figure": "Fig3a", "burnup": 1.1, "series": "100 kW/m", "T": 1402.0, "swelling": 0.58},
    {"figure": "Fig3a", "burnup": 1.1, "series": "100 kW/m", "T": 1485.0, "swelling": 1.22},
    {"figure": "Fig3a", "burnup": 1.1, "series": "100 kW/m", "T": 1549.0, "swelling": 1.84},
    {"figure": "Fig3a", "burnup": 1.1, "series": "100 kW/m", "T": 1598.0, "swelling": 1.66},
    {"figure": "Fig3a", "burnup": 1.1, "series": "100 kW/m", "T": 1632.0, "swelling": 2.13},
    {"figure": "Fig3a", "burnup": 1.1, "series": "100 kW/m", "T": 1669.0, "swelling": 3.60},
    {"figure": "Fig3a", "burnup": 1.1, "series": "100 kW/m", "T": 1685.0, "swelling": 2.72},
    {"figure": "Fig3a", "burnup": 1.1, "series": "119 kW/m", "T": 899.0, "swelling": 0.63},
    {"figure": "Fig3a", "burnup": 1.1, "series": "119 kW/m", "T": 1154.0, "swelling": 1.17},
    {"figure": "Fig3a", "burnup": 1.1, "series": "119 kW/m", "T": 1228.0, "swelling": 1.08},
    {"figure": "Fig3a", "burnup": 1.1, "series": "119 kW/m", "T": 1325.0, "swelling": 1.28},
    {"figure": "Fig3a", "burnup": 1.1, "series": "119 kW/m", "T": 1435.0, "swelling": 1.32},
    {"figure": "Fig3a", "burnup": 1.1, "series": "119 kW/m", "T": 1514.0, "swelling": 2.10},
    {"figure": "Fig3a", "burnup": 1.1, "series": "119 kW/m", "T": 1570.0, "swelling": 2.72},
    {"figure": "Fig3a", "burnup": 1.1, "series": "119 kW/m", "T": 1608.0, "swelling": 2.91},
    {"figure": "Fig3a", "burnup": 1.1, "series": "119 kW/m", "T": 1635.0, "swelling": 3.28},
    {"figure": "Fig3a", "burnup": 1.1, "series": "119 kW/m", "T": 1656.0, "swelling": 3.75},
    {"figure": "Fig3b", "burnup": 1.3, "series": "measurement", "T": 1044.0, "swelling": 1.11},
    {"figure": "Fig3b", "burnup": 1.3, "series": "measurement", "T": 1220.0, "swelling": 1.22},
    {"figure": "Fig3b", "burnup": 1.3, "series": "measurement", "T": 1377.0, "swelling": 1.33},
    {"figure": "Fig3b", "burnup": 1.3, "series": "measurement", "T": 1534.0, "swelling": 2.83},
    {"figure": "Fig3b", "burnup": 1.3, "series": "measurement", "T": 1595.0, "swelling": 3.53},
    {"figure": "Fig3b", "burnup": 1.3, "series": "measurement", "T": 1639.0, "swelling": 3.86},
    {"figure": "Fig3b", "burnup": 1.3, "series": "measurement", "T": 1661.0, "swelling": 2.45},
    {"figure": "Fig3b", "burnup": 1.3, "series": "measurement", "T": 1709.0, "swelling": 2.93},
    {"figure": "Fig3b", "burnup": 1.3, "series": "measurement", "T": 1724.0, "swelling": 3.15},
    {"figure": "Fig3c", "burnup": 3.2, "series": "measurement", "T": 984.0, "swelling": 0.72},
    {"figure": "Fig3c", "burnup": 3.2, "series": "measurement", "T": 1056.0, "swelling": 1.06},
    {"figure": "Fig3c", "burnup": 3.2, "series": "measurement", "T": 1126.0, "swelling": 1.26},
    {"figure": "Fig3c", "burnup": 3.2, "series": "measurement", "T": 1247.0, "swelling": 1.58},
    {"figure": "Fig3c", "burnup": 3.2, "series": "measurement", "T": 1343.0, "swelling": 1.79},
    {"figure": "Fig3c", "burnup": 3.2, "series": "measurement", "T": 1420.0, "swelling": 2.08},
    {"figure": "Fig3c", "burnup": 3.2, "series": "measurement", "T": 1459.0, "swelling": 2.40},
    {"figure": "Fig3c", "burnup": 3.2, "series": "measurement", "T": 1511.0, "swelling": 2.83},
    {"figure": "Fig3c", "burnup": 3.2, "series": "measurement", "T": 1557.0, "swelling": 3.31},
    {"figure": "Fig3c", "burnup": 3.2, "series": "measurement", "T": 1590.0, "swelling": 3.75},
]

EXP_SWELLING_BURNUP_1600 = [
    {"burnup": 1.12, "swelling": 1.64, "series": "approx 100 kW/m"},
    {"burnup": 1.11, "swelling": 2.90, "series": "approx 119 kW/m"},
    {"burnup": 1.31, "swelling": 3.51, "series": "measurement"},
    {"burnup": 3.18, "swelling": 3.72, "series": "measurement"},
]

EXP_ND_T_13 = [
    {"T": 1153.0, "N": 5.22e19},
    {"T": 1202.0, "N": 3.69e19},
    {"T": 1227.0, "N": 3.00e19},
    {"T": 1235.0, "N": 3.57e19},
    {"T": 1248.0, "N": 2.44e19},
    {"T": 1330.0, "N": 2.90e19},
    {"T": 1338.0, "N": 4.10e19},
    {"T": 1377.0, "N": 2.71e19},
    {"T": 1408.0, "N": 1.67e19},
    {"T": 1427.0, "N": 1.85e19},
    {"T": 1493.0, "N": 3.00e19},
    {"T": 1507.0, "N": 2.44e19},
    {"T": 1524.0, "N": 1.56e19},
    {"T": 1538.0, "N": 7.54e18},
    {"T": 1555.0, "N": 7.80e18},
    {"T": 1561.0, "N": 1.92e19},
    {"T": 1599.0, "N": 5.34e18},
    {"T": 1622.0, "N": 7.80e18},
    {"T": 1628.0, "N": 1.73e19},
    {"T": 1649.0, "N": 5.34e18},
    {"T": 1656.0, "N": 3.65e18},
    {"T": 1656.0, "N": 2.41e18},
    {"T": 1669.0, "N": 1.96e18},
    {"T": 1682.0, "N": 2.44e19},
    {"T": 1685.0, "N": 1.40e19},
    {"T": 1723.0, "N": 7.70e17},
    {"T": 1742.0, "N": 1.02e18},
    {"T": 1739.0, "N": 1.65e18},
]

EXP_RD_T_13 = [
    {"T": 1045.0, "R_nm": 54.06},
    {"T": 1219.0, "R_nm": 60.37},
    {"T": 1374.0, "R_nm": 69.58},
    {"T": 1535.0, "R_nm": 104.85},
    {"T": 1594.0, "R_nm": 120.83},
    {"T": 1641.0, "R_nm": 143.73},
    {"T": 1663.0, "R_nm": 122.76},
    {"T": 1710.0, "R_nm": 157.99},
    {"T": 1725.0, "R_nm": 173.67},
]


## Solver core

This is the Zullo `un_model.py` solver embedded in the notebook. The top settings cell controls flags, constants, and A20.

In [129]:
import math
from dataclasses import dataclass
from typing import Optional, Sequence, Dict, List, Tuple

# Solver core from Zullo's un_model.py, embedded for standalone use.
# Main switches/constants are intentionally defined in the first notebook cell.

# ============================================================
# DATACLASSES
# ============================================================

@dataclass(frozen=True)
class Candidate:
    label: str
    f_n: float
    K_d: float
    rho_d: float
    fission_rate: float

    Dv_scale: float = 1.0
    Dv_D1_scale: float = 1.0
    Dv_D2_scale: float = 1.0
    Dv_dislocation_scale: float = 1.0

    Dg_scale: float = 1.0
    b_scale: float = 1.0
    gb_scale: float = 1.0
    gd_scale: float = 1.0
    coalescence_d_scale: float = 1.0
    capture_scale: float = 1.0

    D2_xe_scale: float = 1.0

    Dg_D1_scale: float = 1.0
    Dg_D3_scale: float = 1.0
    Dg_dislocation_scale: float = 1.0

    b_bulk_scale: float = 1.0
    b_dislocation_scale: float = 1.0

    gd_bubble_scale: float = 1.0
    gd_line_scale: float = 1.0
    gd_line_alpha: float = 1.0


@dataclass
class UNParameters:
    temperature: float = 1600.0
    fission_rate: float = FISSION_RATE_NOMINAL
    grain_radius: float = GRAIN_RADIUS
    target_burnup_percent_fima: Optional[float] = None
    final_time: float = 24.0 * 3600.0
    dt: float = 3600.0
    n_modes: int = 40

    xe_yield: float = XE_YIELD
    precursor_factor: float = 1.0

    D10: float = 1.56e-3
    Q1: float = 4.94
    A20_xe: float = 1.21e-67
    B21_xe: float = 25.87
    B22_xe: float = -1.49
    B23_xe: float = 0.0
    A30: float = 1.85e-39
    D2_xe_scale: float = 1.0
    Dg_scale: float = 1.0
    Dg_D1_scale: float = 1.0
    Dg_D3_scale: float = 1.0
    Dg_dislocation_scale: float = 1.0
    xe_diffusivity_mode: str = XE_DIFFUSIVITY_MODE

    kB_eV: float = 8.617333262e-5
    kB_J: float = 1.380649e-23

    D10_vU: float = 1.35e-2
    Q1_vU: float = 5.66
    B21_vU: float = -0.62
    B22_vU: float = -0.04
    A20_vU: float = A20_VU_DEFAULT
    Dv_scale: float = 1.0
    Dv_D1_scale: float = 1.0
    Dv_D2_scale: float = 1.0
    Dv_dislocation_scale: float = 1.0
    vacancy_diffusivity_mode: str = VU_DIFFUSIVITY_MODE

    radius_in_lattice: float = 0.21e-9
    omega_fg: float = OMEGA_FG
    lattice_parameter: float = LATTICE_PARAMETER
    gamma_b: float = GAMMA_B
    hydrostatic_stress: float = 0.0
    min_radius_for_pressure: float = 1.0e-15

    f_n: float = F_N_NOMINAL
    rho_d: float = RHO_D_NOMINAL
    K_d: float = K_D_NOMINAL
    r_d: float = 3.46e-10
    Z_d: float = 5.0

    Dg_extra_scale: float = 1.0
    gb_scale: float = 1.0
    gd_scale: float = 1.0
    b_scale: float = 1.0
    b_bulk_scale: float = 1.0
    b_dislocation_scale: float = 1.0
    gd_bubble_scale: float = 1.0
    gd_line_scale: float = 1.0
    gd_line_alpha: float = 1.0
    coalescence_d_scale: float = 1.0
    capture_scale: float = 1.0

    R_b: float = 0.0
    N_b: float = 0.0
    R_d: float = 0.0
    N_d: Optional[float] = None
    c0: float = 0.0
    mb0: float = 0.0
    md0: float = 0.0
    nvb0: Optional[float] = None
    nvd0: Optional[float] = None

    bulk_seed_radius_nm: float = 0.0
    vacancy_absorption_only: bool = True
    update_bulk_vacancies: bool = True
    min_number_density: float = 0.0
    min_volume: float = 0.0

    def __post_init__(self):
        if self.N_d is None:
            self.N_d = self.K_d * self.rho_d
        if self.target_burnup_percent_fima is not None:
            self.final_time = burnup_percent_to_time(
                self.target_burnup_percent_fima,
                self.fission_rate,
                self.lattice_parameter,
            )

# ============================================================
# HELPER FUNCTIONS
# ============================================================

def omega_matrix(p: UNParameters) -> float:
    return p.lattice_parameter**3 / 4.0


def uranium_atom_density_from_lattice(lattice_parameter: float) -> float:
    return 4.0 / lattice_parameter**3


def burnup_percent_to_time(burnup_percent_fima: float, fission_rate: float, lattice_parameter: float) -> float:
    if fission_rate <= 0.0:
        raise ValueError("fission_rate must be positive")
    return (burnup_percent_fima / 100.0) * uranium_atom_density_from_lattice(lattice_parameter) / fission_rate


def time_to_burnup_percent(time: float, fission_rate: float, lattice_parameter: float) -> float:
    return 100.0 * fission_rate * time / uranium_atom_density_from_lattice(lattice_parameter)


def sphere_volume(R: float) -> float:
    return 0.0 if R <= 0.0 else (4.0 / 3.0) * math.pi * R**3


def radius_from_volume(V: float) -> float:
    return 0.0 if V <= 0.0 else (3.0 * V / (4.0 * math.pi)) ** (1.0 / 3.0)


def _safe_exp(x: float) -> float:
    return math.exp(max(min(x, 700.0), -745.0))


def xe_diffusivity_UN(p: UNParameters):
    """Xe / fission-gas diffusivity with selectable Rizk 2025/2023 modes.

    Modes are selected in the first notebook cell through XE_DIFFUSIVITY_MODE.
    The default mode reproduces the previous code: Rizk 2025 D1 + D3, with Xe D2 neglected.
    """
    T = p.temperature
    kBT = p.kB_eV * T
    mode = p.xe_diffusivity_mode

    # Initialize all diagnostics to zero for a consistent output dictionary.
    D1 = D2 = D3 = 0.0
    D1_Xe_i = D2_Xe_i_0 = D2_Xe_i_1 = 0.0
    D1_XeVU = D2_XeVU = 0.0

    if mode == "rizk2025_table_D1D3":
        # Rizk 2025 Table 2. D2 for Xe is computed only as a diagnostic and intentionally neglected.
        D1 = p.D10 * math.exp(-p.Q1 / kBT)
        D2 = math.sqrt(p.fission_rate) * p.A20_xe * _safe_exp(
            -p.B21_xe / kBT - p.B22_xe / (kBT**2) - p.B23_xe / (kBT**3)
        )
        D3 = p.A30 * p.fission_rate
        Dg_unscaled = p.Dg_D1_scale * D1 + p.Dg_D3_scale * D3
        D2_scaled = 0.0

    elif mode == "rizk2025_refit_plot":
        # Refit of the digitized Rizk 2025 D_gas curve at Fdot = 1e19.
        D10 = 2.967341817979e-03
        Q1 = 5.013616576464e+00
        A20 = 4.498475254045e-68
        B1 = -1.790671812685e+01
        B2 = 9.255235831189e-01
        A30 = 1.189275430019e-38
        D1 = D10 * math.exp(-Q1 / kBT)
        D2 = math.sqrt(p.fission_rate) * A20 * _safe_exp(-B1 / kBT - B2 / (kBT**2))
        D3 = A30 * p.fission_rate
        D2_scaled = p.D2_xe_scale * D2
        Dg_unscaled = p.Dg_D1_scale * D1 + D2_scaled + p.Dg_D3_scale * D3

    elif mode == "rizk2023_table_raw":
        # Rizk 2023 Table 3.1 raw Xe diffusivity: Xei + {Xe:VU} + athermal D3.
        # Use Dg_scale = 5 if you want the calibrated Rizk 2023 model multiplier from Table 3.4.
        D1_Xe_i = 6.12e-5 * math.exp(-5.76 / kBT)
        D2_Xe_i_0 = math.sqrt(p.fission_rate) * 3.40e-12 * _safe_exp(
            -27.67 / kBT - (-5.28) / (kBT**2) - 0.302 / (kBT**3)
        )
        D2_Xe_i_1 = math.sqrt(p.fission_rate) * 1.45e-33 * _safe_exp(
            -(-1.80) / kBT - 0.156 / (kBT**2)
        )
        D3 = 8.2e-42 * p.fission_rate
        D1_XeVU = 7.54e-4 * math.exp(-5.76 / kBT)
        D2_XeVU = math.sqrt(p.fission_rate) * 2.06e-33 * _safe_exp(
            -(-0.408) / kBT - (-0.0798) / (kBT**2) - 0.00940 / (kBT**3)
        )
        D1 = D1_Xe_i + D1_XeVU
        D2 = D2_Xe_i_0 + D2_Xe_i_1 + D2_XeVU
        D2_scaled = p.D2_xe_scale * D2
        Dg_unscaled = p.Dg_D1_scale * D1 + D2_scaled + p.Dg_D3_scale * D3

    else:
        raise ValueError(
            f"Unknown xe_diffusivity_mode={mode!r}. "
            "Use 'rizk2025_table_D1D3', 'rizk2025_refit_plot', or 'rizk2023_table_raw'."
        )

    Dg = Dg_unscaled * p.Dg_scale * p.precursor_factor * p.Dg_extra_scale
    return Dg, {
        "xe_diffusivity_mode": mode,
        "D1_Xe": D1,
        "D2_Xe": D2,
        "D2_Xe_scaled": D2_scaled,
        "D3_Xe": D3,
        "D1_Xe_i_2023": D1_Xe_i,
        "D2_Xe_i0_2023": D2_Xe_i_0,
        "D2_Xe_i1_2023": D2_Xe_i_1,
        "D1_XeVU_2023": D1_XeVU,
        "D2_XeVU_2023": D2_XeVU,
        "Dg_D1_scaled": p.Dg_D1_scale * D1,
        "Dg_D3_scaled": p.Dg_D3_scale * D3,
        "D2_Xe_over_Dg_unscaled": (D2 / Dg_unscaled) if Dg_unscaled > 0 and math.isfinite(Dg_unscaled) else math.nan,
        "Dg": Dg,
        "Dg_dislocation_scale": p.Dg_dislocation_scale,
        "Dg_dislocation": Dg * p.Dg_dislocation_scale,
    }


def vacancy_diffusivity_UN(p: UNParameters):
    """U-vacancy diffusivity with selectable Rizk 2025/2023 modes."""
    T = p.temperature
    kBT = p.kB_eV * T
    mode = p.vacancy_diffusivity_mode

    if mode == "rizk2025_code_refit_old":
        # Previous code behavior: Zullo/v14 sign convention and old A20 refit.
        D1 = p.D10_vU * math.exp(-p.Q1_vU / kBT)
        D2 = math.sqrt(p.fission_rate) * p.A20_vU * _safe_exp(
            p.B21_vU / kBT + p.B22_vU / (kBT**2)
        )

    elif mode == "rizk2025_refit_A20_only":
        # Same D1, Q1, B1, B2 as the previous code; only A20 is refitted to the digitized Rizk 2025 curve.
        D1 = 1.35e-2 * math.exp(-5.66 / kBT)
        D2 = math.sqrt(p.fission_rate) * 1.386341579723e-28 * _safe_exp(
            -0.62 / kBT - 0.04 / (kBT**2)
        )

    elif mode == "rizk2025_refit_full":
        # Full refit to the digitized Rizk 2025 U-vacancy curve.
        D1 = 1.122978768506e-02 * math.exp(-5.596873538604e+00 / kBT)
        D2 = math.sqrt(p.fission_rate) * 7.805188680989e-28 * _safe_exp(
            -9.932675113163e-01 / kBT - 2.082395503235e-02 / (kBT**2)
        )

    elif mode == "rizk2023_table_raw":
        # Rizk 2023 Table 3.1 raw V_U diffusivity.
        # Use Dv_scale = 5 if you want the calibrated Rizk 2023 model multiplier from Table 3.4.
        D1 = 2.11e-4 * math.exp(-6.04 / kBT)
        D2 = math.sqrt(p.fission_rate) * 8.12e-42 * _safe_exp(
            -(-6.01) / kBT - 0.543 / (kBT**2) - (-0.0124) / (kBT**3)
        )

    else:
        raise ValueError(
            f"Unknown vacancy_diffusivity_mode={mode!r}. "
            "Use 'rizk2025_code_refit_old', 'rizk2025_refit_A20_only', "
            "'rizk2025_refit_full', or 'rizk2023_table_raw'."
        )

    Dv_unscaled = p.Dv_D1_scale * D1 + p.Dv_D2_scale * D2
    Dv = Dv_unscaled * p.Dv_scale
    Dv_dislocation = Dv * p.Dv_dislocation_scale
    return Dv, {
        "vacancy_diffusivity_mode": mode,
        "Dv1": D1,
        "Dv2": D2,
        "Dv1_scaled": p.Dv_D1_scale * D1,
        "Dv2_scaled": p.Dv_D2_scale * D2,
        "Dv": Dv,
        "Dv_dislocation_scale": p.Dv_dislocation_scale,
        "Dv_dislocation": Dv_dislocation,
        "A20_vU_active": p.A20_vU if mode == "rizk2025_code_refit_old" else math.nan,
    }

def b0_resolution(R: float) -> float:
    R = max(R, 1.0e-15)
    return 1.0e-25 * (2.64 - 2.02 * math.exp(-2.61e-9 / R))


def resolution_rates_UN(p: UNParameters, R_b: float, R_d: float):
    b_b = p.fission_rate * b0_resolution(R_b + p.radius_in_lattice) * p.b_scale * p.b_bulk_scale
    b_d = p.fission_rate * b0_resolution(R_d + p.radius_in_lattice) * p.b_scale * p.b_dislocation_scale
    return b_b, b_d


def trapping_rates_UN(p: UNParameters, Dg: float, R_b: float, N_b: float, R_d: float, N_d: float):
    Rb_eff = R_b + p.radius_in_lattice
    Rd_eff = R_d + p.radius_in_lattice
    g_b_unscaled = 0.0 if N_b <= 0.0 else 4.0 * math.pi * Dg * Rb_eff * N_b

    Gamma_d = 1.0 / math.sqrt(math.pi * p.rho_d)
    den = math.log(Gamma_d / (p.Z_d * p.r_d)) - 3.0 / 5.0
    if den <= 0.0:
        raise ValueError(f"Invalid dislocation sink denominator: {den:g}")

    free_dislocation = max(p.rho_d - p.gd_line_alpha * 2.0 * R_d * N_d, 0.0)
    Dg_dislocation = Dg * p.Dg_dislocation_scale
    term_bubbles = 4.0 * math.pi * Dg * Rd_eff * N_d
    term_dislocation = (2.0 * math.pi * Dg_dislocation / den) * free_dislocation
    g_d_unscaled = p.gd_bubble_scale * term_bubbles + p.gd_line_scale * term_dislocation

    g_b = p.gb_scale * g_b_unscaled
    g_d = p.gd_scale * g_d_unscaled

    return g_b, g_d, {
        "Gamma_d": Gamma_d,
        "den": den,
        "free_dislocation": free_dislocation,
        "term_bubbles": term_bubbles,
        "term_dislocation": term_dislocation,
        "Dg_dislocation_scale": p.Dg_dislocation_scale,
        "Dg_dislocation": Dg_dislocation,
        "term_bubbles_scaled": p.gd_bubble_scale * term_bubbles,
        "term_dislocation_scaled": p.gd_line_scale * term_dislocation,
        "g_b_unscaled": g_b_unscaled,
        "g_d_unscaled": g_d_unscaled,
    }


def beta_production(p: UNParameters) -> float:
    return p.xe_yield * p.fission_rate


def nucleation_rate_bulk(p: UNParameters, Dg: float, c: float) -> float:
    return 8.0 * math.pi * p.f_n * Dg * p.omega_fg ** (1.0 / 3.0) * max(c, 0.0) ** 2


def phi_population(m_gas: float, N: float) -> float:
    if N <= 0.0 or m_gas <= 0.0:
        return 0.0
    atoms_per_bubble = m_gas / N
    if atoms_per_bubble <= 1.0:
        return 0.0
    return 1.0 / (atoms_per_bubble - 1.0)


def coalescence_lambda(Vd: float, Nd: float) -> float:
    xi = max(0.0, min(Vd * Nd, 0.999999))
    return (2.0 - xi) / (2.0 * (1.0 - xi) ** 3)


def pressure_internal(p: UNParameters, m_gas: float, n_vac: float) -> float:
    if m_gas <= 0.0:
        return 0.0
    if n_vac <= 0.0:
        return math.inf
    denom = n_vac * omega_matrix(p)
    return math.inf if denom <= 0.0 else p.kB_J * p.temperature * m_gas / denom


def pressure_equilibrium(p: UNParameters, R: float) -> float:
    return 2.0 * p.gamma_b / max(R, p.min_radius_for_pressure) - p.hydrostatic_stress


def seed_vacancies_for_pressure_factor(
    p: UNParameters,
    gas_atoms_per_bubble: float,
    pressure_factor: float,
) -> Tuple[float, float]:
    """Return (vacancies_per_bubble, radius) for a gas dimer seed at p = factor * p_eq.

    Per-bubble equations:
        p = kBT * n_g / (n_v * Omega)
        p_eq = 2 gamma / R
        R = [3*(Omega_fg*n_g + Omega*n_v)/(4*pi)]^(1/3)
        p = pressure_factor * p_eq

    The equation is solved by robust bisection in n_v.
    """
    ng = max(float(gas_atoms_per_bubble), 1.0e-30)
    f = max(float(pressure_factor), 1.0e-12)
    Om = omega_matrix(p)

    def radius_from_nv(nv: float) -> float:
        V = p.omega_fg * ng + Om * nv
        return radius_from_volume(max(V, 1.0e-300))

    def residual(nv: float) -> float:
        R = radius_from_nv(nv)
        p_int = p.kB_J * p.temperature * ng / (nv * Om)
        p_eq = 2.0 * p.gamma_b / max(R, p.min_radius_for_pressure) - p.hydrostatic_stress
        return p_int - f * p_eq

    lo = 1.0e-30
    hi = 1.0
    # Increase upper bound until pressure is below target.
    for _ in range(300):
        if residual(hi) < 0.0:
            break
        hi *= 2.0
    else:
        # Fallback: very large vacancy count if no bracket is found.
        R = radius_from_nv(hi)
        return hi, R

    for _ in range(160):
        mid = math.sqrt(lo * hi)
        if residual(mid) > 0.0:
            lo = mid
        else:
            hi = mid

    nv = hi
    return nv, radius_from_nv(nv)


def dimer_seed_for_population(p: UNParameters, N: float) -> Tuple[float, float, float]:
    """Return (m_seed, n_v_seed, R_seed) for N bubbles."""
    if N <= 0.0 or not USE_EQUILIBRIUM_DIMER_SEED:
        return 0.0, 0.0, 0.0
    nv_per_bubble, R_seed = seed_vacancies_for_pressure_factor(
        p,
        SEED_GAS_ATOMS_PER_BUBBLE,
        SEED_PRESSURE_FACTOR,
    )
    m_seed = SEED_GAS_ATOMS_PER_BUBBLE * N
    n_v_seed = nv_per_bubble * N
    return m_seed, n_v_seed, R_seed


def vacancies_for_pressure_factor_from_gas(
    p: UNParameters,
    m_gas: float,
    N: float,
    pressure_factor: float,
) -> Tuple[float, float]:
    """Return (total_vacancy_concentration, radius) for existing gas m_gas in N bubbles.

    It solves per bubble:
        p = kBT * n_g / (n_v * Omega)
        p_eq = 2 gamma / R
        R = [3*(Omega_fg*n_g + Omega*n_v)/(4*pi)]^(1/3)
        p = pressure_factor * p_eq
    where n_g = m_gas/N and n_v is vacancies per bubble.
    """
    if m_gas <= 0.0 or N <= 0.0:
        return 0.0, 0.0
    gas_atoms_per_bubble = max(m_gas / N, 1.0e-30)
    nv_per_bubble, R_seed = seed_vacancies_for_pressure_factor(
        p,
        gas_atoms_per_bubble,
        pressure_factor,
    )
    return nv_per_bubble * N, R_seed


def gas_only_radius_for_population(p: UNParameters, m_gas: float, N: float) -> float:
    if m_gas <= 0.0 or N <= 0.0:
        return 0.0
    return radius_from_volume(p.omega_fg * m_gas / N)


def radius_for_vacancy_update(p: UNParameters, R_old: float, N: float, m_gas: float) -> float:
    if R_old > 0.0:
        return R_old
    return gas_only_radius_for_population(p, m_gas, N)


def wigner_seitz_delta(N: float) -> float:
    return (3.0 / (4.0 * math.pi * max(N, 1.0))) ** (1.0 / 3.0)


def zeta_geometry(R: float, N: float) -> float:
    delta = wigner_seitz_delta(N)
    psi = max(R / delta, 1.0e-12)

    if not math.isfinite(psi):
        return 1.0e300

    if USE_ZETA_OVERFLOW_GUARD and psi >= 1.0:
        # In the original formula the denominator becomes non-positive for large psi
        # and is clipped to 1e-30. Avoid evaluating psi**6 when psi is enormous.
        if psi > 1.0e75:
            return 1.0e300
        return max(10.0 * psi * (1.0 + psi**3) / 1.0e-30, 1.0e-30)

    den = -psi**6 + 5.0 * psi**2 - 9.0 * psi + 5.0
    den = max(den, 1.0e-30)
    return max(10.0 * psi * (1.0 + psi**3) / den, 1.0e-30)


def vacancy_concentration_implicit_step(p: UNParameters, Dv: float, R: float, N: float, m_gas: float, n_old: float, dt: float):
    if N <= 0.0 or m_gas <= 0.0:
        return n_old, 0.0
    R_update = radius_for_vacancy_update(p, R, N, m_gas)
    if R_update <= 0.0:
        return n_old, 0.0

    p_eq = 2.0 * p.gamma_b / R_update - p.hydrostatic_stress
    p_int_old = pressure_internal(p, m_gas, n_old)

    if p.vacancy_absorption_only and p_int_old <= p_eq:
        return n_old, 0.0

    delta = wigner_seitz_delta(N)
    zeta = zeta_geometry(R_update, N)
    A = 2.0 * math.pi * Dv * delta * N / (p.kB_J * p.temperature * zeta)
    C = p.kB_J * p.temperature * m_gas / omega_matrix(p)
    B = n_old - dt * A * p_eq
    disc = B * B + 4.0 * dt * A * C

    if disc < 0.0:
        raise ValueError(f"Negative discriminant in vacancy step: {disc:g}")

    sqrt_disc = math.sqrt(disc)
    if B >= 0.0:
        n_new = 0.5 * (B + sqrt_disc)
    else:
        denom = sqrt_disc - B
        n_new = 0.0 if denom <= 0.0 else (2.0 * dt * A * C) / denom

    if p.vacancy_absorption_only:
        n_new = max(n_new, n_old)

    return n_new, (n_new - n_old) / dt


def initialize_vacancy_concentration(p: UNParameters, N: float, R: float, m_gas: float) -> float:
    if N <= 0.0 or R <= 0.0:
        return 0.0
    vacancy_volume = max(N * sphere_volume(R) - p.omega_fg * m_gas, 0.0)
    return vacancy_volume / omega_matrix(p)


def initialize_modes_from_average(average: float, n_modes: int, n_iter: int = 20):
    modes = [0.0 for _ in range(n_modes)]
    projection_coeff = -math.sqrt(8.0 / math.pi)
    remainder = average
    for _ in range(n_iter):
        reconstructed = 0.0
        for i in range(n_modes):
            n = i + 1
            n_coeff = (-1.0) ** n / n
            modes[i] += projection_coeff * n_coeff * remainder
            reconstructed += projection_coeff * n_coeff * modes[i] * 3.0 / (4.0 * math.pi)
        remainder = average - reconstructed
    return modes


def reconstruct_average(modes: Sequence[float]) -> float:
    projection_coeff = -2.0 * math.sqrt(2.0 / math.pi)
    average = 0.0
    for i, value in enumerate(modes):
        n = i + 1
        n_coeff = (-1.0) ** n / n
        average += projection_coeff * n_coeff * value / ((4.0 / 3.0) * math.pi)
    return average


def det3(A):
    return (
        A[0][0] * (A[1][1] * A[2][2] - A[1][2] * A[2][1])
        - A[0][1] * (A[1][0] * A[2][2] - A[1][2] * A[2][0])
        + A[0][2] * (A[1][0] * A[2][1] - A[1][1] * A[2][0])
    )


def solve3x3_cramer(A, b):
    detA = det3(A)
    if abs(detA) < 1.0e-300:
        raise ZeroDivisionError("Singular 3x3 system")
    Ax = [[b[i], A[i][1], A[i][2]] for i in range(3)]
    Ay = [[A[i][0], b[i], A[i][2]] for i in range(3)]
    Az = [[A[i][0], A[i][1], b[i]] for i in range(3)]
    return [det3(Ax) / detA, det3(Ay) / detA, det3(Az) / detA]


def sciantix_3x3_exchange_step(
    modes_c,
    modes_mb,
    modes_md,
    Dg: float,
    R_grain: float,
    source_c: float,
    source_mb: float,
    source_md: float,
    g_b: float,
    g_d: float,
    b_b_gas: float,
    b_d_gas: float,
    dt: float,
):
    projection_coeff = -2.0 * math.sqrt(2.0 / math.pi)
    diffusion_rate_coeff = math.pi**2 * Dg / R_grain**2

    for i in range(len(modes_c)):
        n = i + 1
        n_coeff = (-1.0) ** n / n
        diffusion_rate = diffusion_rate_coeff * n**2

        src_c = projection_coeff * source_c * n_coeff
        src_mb = projection_coeff * source_mb * n_coeff
        src_md = projection_coeff * source_md * n_coeff

        A = [
            [1.0 + (diffusion_rate + g_b + g_d) * dt, -b_b_gas * dt, -b_d_gas * dt],
            [-g_b * dt, 1.0 + b_b_gas * dt, 0.0],
            [-g_d * dt, 0.0, 1.0 + b_d_gas * dt],
        ]
        rhs = [
            modes_c[i] + src_c * dt,
            modes_mb[i] + src_mb * dt,
            modes_md[i] + src_md * dt,
        ]
        modes_c[i], modes_mb[i], modes_md[i] = solve3x3_cramer(A, rhs)

    return reconstruct_average(modes_c), reconstruct_average(modes_mb), reconstruct_average(modes_md)


def reset_modes_to_averages(c: float, mb: float, md: float, n_modes: int):
    return (
        initialize_modes_from_average(max(c, 0.0), n_modes),
        initialize_modes_from_average(max(mb, 0.0), n_modes),
        initialize_modes_from_average(max(md, 0.0), n_modes),
    )

# ============================================================
# SOLVER
# ============================================================

def solve_UN(p: UNParameters, keep_history: bool = True):
    # Initial modes are set after optional seed initialization below.
    modes_c = None
    modes_mb = None
    modes_md = None

    R_b = p.R_b
    R_d = p.R_d
    N_b = p.N_b
    N_d = p.N_d

    c0_eff = p.c0
    mb0_eff = p.mb0
    md0_eff = p.md0

    nvb = initialize_vacancy_concentration(p, N_b, R_b, mb0_eff) if p.nvb0 is None else p.nvb0
    nvd = initialize_vacancy_concentration(p, N_d, R_d, md0_eff) if p.nvd0 is None else p.nvd0

    # Optional initial dimer seed for dislocation bubbles:
    # N_d exists at t=0; if R_d=0 and md0=nvd0=0, seed each site as a dimer at p = factor*Peq.
    if (
        USE_EQUILIBRIUM_DIMER_SEED
        and SEED_INITIAL_DISLOCATION_BUBBLES
        and N_d > 0.0
        and R_d <= 0.0
        and md0_eff <= 0.0
        and nvd <= 0.0
    ):
        md0_eff, nvd, R_d = dimer_seed_for_population(p, N_d)

    V_b = sphere_volume(R_b)
    V_d = sphere_volume(R_d)

    modes_c, modes_mb, modes_md = reset_modes_to_averages(c0_eff, mb0_eff, md0_eff, p.n_modes)

    beta = beta_production(p)
    initial_gas = c0_eff + mb0_eff + md0_eff
    generated = 0.0
    q_gb = 0.0
    retained = initial_gas
    t = 0.0

    capture_fraction_sum = 0.0
    capture_raw_sum = 0.0
    capture_bubbles_cumulative = 0.0
    max_f_cap_step = 0.0

    bulk_first_gas_seed_done = False
    dislocation_first_gas_seed_done = False

    hist_keys = [
        "time", "burnup_percent_fima", "c", "mb", "md", "Nb", "Nd", "Vb", "Vd", "Rb", "Rd",
        "nvb", "nvd", "generated", "retained", "q_gb", "swelling_b", "swelling_d", "swelling_ig",
        "p_b", "p_d", "p_b_eq", "p_d_eq", "lambda_d", "nu_b", "phi_b", "phi_d",
        "f_cap_step", "cap_raw_step", "capture_fraction_sum", "capture_raw_sum",
        "capture_bubbles_cumulative", "max_f_cap_step",
        "matrix_gas_percent", "bulk_gas_percent", "dislocation_gas_percent", "qgb_gas_percent",
    ]
    hist = {key: [] for key in hist_keys}

    def append_state(nu_b=0.0, phi_b=0.0, phi_d=0.0, lambda_d=0.0, fcap=0.0, capraw=0.0):
        if not keep_history:
            return
        c_av = reconstruct_average(modes_c)
        mb_av = reconstruct_average(modes_mb)
        md_av = reconstruct_average(modes_md)
        p_b = pressure_internal(p, mb_av, nvb)
        p_d = pressure_internal(p, md_av, nvd)
        p_b_eq = pressure_equilibrium(p, R_b)
        p_d_eq = pressure_equilibrium(p, R_d)

        hist["time"].append(t)
        hist["burnup_percent_fima"].append(time_to_burnup_percent(t, p.fission_rate, p.lattice_parameter))
        hist["c"].append(c_av)
        hist["mb"].append(mb_av)
        hist["md"].append(md_av)
        hist["Nb"].append(N_b)
        hist["Nd"].append(N_d)
        hist["Vb"].append(V_b)
        hist["Vd"].append(V_d)
        hist["Rb"].append(R_b)
        hist["Rd"].append(R_d)
        hist["nvb"].append(nvb)
        hist["nvd"].append(nvd)
        hist["generated"].append(generated)
        hist["retained"].append(retained)
        hist["q_gb"].append(q_gb)
        hist["swelling_b"].append(N_b * V_b)
        hist["swelling_d"].append(N_d * V_d)
        hist["swelling_ig"].append(N_b * V_b + N_d * V_d)
        hist["p_b"].append(p_b)
        hist["p_d"].append(p_d)
        hist["p_b_eq"].append(p_b_eq)
        hist["p_d_eq"].append(p_d_eq)
        hist["lambda_d"].append(lambda_d)
        hist["nu_b"].append(nu_b)
        hist["phi_b"].append(phi_b)
        hist["phi_d"].append(phi_d)
        hist["f_cap_step"].append(fcap)
        hist["cap_raw_step"].append(capraw)
        hist["capture_fraction_sum"].append(capture_fraction_sum)
        hist["capture_raw_sum"].append(capture_raw_sum)
        hist["capture_bubbles_cumulative"].append(capture_bubbles_cumulative)
        hist["max_f_cap_step"].append(max_f_cap_step)
        hist["matrix_gas_percent"].append(100.0 * c_av / generated if generated > 0.0 else 0.0)
        hist["bulk_gas_percent"].append(100.0 * mb_av / generated if generated > 0.0 else 0.0)
        hist["dislocation_gas_percent"].append(100.0 * md_av / generated if generated > 0.0 else 0.0)
        hist["qgb_gas_percent"].append(100.0 * q_gb / generated if generated > 0.0 else 0.0)

    append_state()
    last_rates = {}
    n_steps = int(math.ceil(p.final_time / p.dt))

    for _ in range(n_steps):
        dt = min(p.dt, p.final_time - t)
        if dt <= 0.0:
            break

        c_old = reconstruct_average(modes_c)
        mb_old = reconstruct_average(modes_mb)
        md_old = reconstruct_average(modes_md)

        Nb_old = N_b
        Nd_old = N_d
        Vd_old = V_d
        Rb_old = R_b
        Rd_old = R_d

        Dg, D_parts = xe_diffusivity_UN(p)
        Dv, Dv_parts = vacancy_diffusivity_UN(p)
        Dv_d = Dv * p.Dv_dislocation_scale
        b_b, b_d = resolution_rates_UN(p, R_b, R_d)
        g_b, g_d, trapping_parts = trapping_rates_UN(p, Dg, R_b, Nb_old, R_d, Nd_old)

        nu_b = nucleation_rate_bulk(p, Dg, c_old)
        phi_b = phi_population(mb_old, Nb_old)
        phi_d = phi_population(md_old, Nd_old)

        if USE_PHI_GAS_RESOLUTION:
            b_b_gas = b_b * phi_b
            b_d_gas = b_d * phi_d
        else:
            b_b_gas = b_b
            b_d_gas = b_d

        delta_Nb_nucleated = max(dt * nu_b, 0.0)
        N_b = (Nb_old + delta_Nb_nucleated) / (1.0 + dt * b_b * phi_b)
        N_b = max(N_b, p.min_number_density)

        # If seed is active, nucleating a bulk bubble means creating a gas dimer.
        # This is intentionally separate from USE_NUCLEATION_MASS_COUPLING so the
        # seed remains physical even when testing mass-coupling OFF.
        if USE_EQUILIBRIUM_DIMER_SEED and SEED_NEW_BULK_BUBBLES:
            source_c = beta - SEED_GAS_ATOMS_PER_BUBBLE * nu_b
            source_mb = SEED_GAS_ATOMS_PER_BUBBLE * nu_b
        elif USE_NUCLEATION_MASS_COUPLING:
            source_c = beta - 2.0 * nu_b
            source_mb = 2.0 * nu_b
        else:
            source_c = beta
            source_mb = 0.0
        source_md = 0.0

        c_new, mb_new, md_new = sciantix_3x3_exchange_step(
            modes_c, modes_mb, modes_md,
            Dg, p.grain_radius,
            source_c, source_mb, source_md,
            g_b, g_d,
            b_b_gas, b_d_gas,
            dt,
        )

        if c_new < 0.0 or mb_new < 0.0 or md_new < 0.0:
            c_new = max(c_new, 0.0)
            mb_new = max(mb_new, 0.0)
            md_new = max(md_new, 0.0)
            modes_c, modes_mb, modes_md = reset_modes_to_averages(c_new, mb_new, md_new, p.n_modes)

        dmb_dt = (mb_new - mb_old) / dt
        dmd_dt = (md_new - md_old) / dt

        nvb_before_vacancy_update = nvb
        nvd_before_vacancy_update = nvd
        nvb_seed_added = 0.0
        nvd_seed_added = 0.0
        bulk_first_gas_seed_applied = False
        dislocation_first_gas_seed_applied = False

        # Current test: when a population first receives gas, assign vacancies so that
        # the pressure at the end of that first gas step is FIRST_GAS_PRESSURE_FACTOR*p_eq.
        # Then skip normal vacancy absorption for that population in this same step.
        if (
            USE_FIRST_GAS_PRESSURE_SEED
            and SEED_BULK_ON_FIRST_GAS
            and (not bulk_first_gas_seed_done)
            and N_b > 0.0
            and mb_old <= 0.0
            and mb_new > 0.0
        ):
            nvb_target, _Rb_seed_from_gas = vacancies_for_pressure_factor_from_gas(
                p,
                mb_new,
                N_b,
                FIRST_GAS_PRESSURE_FACTOR,
            )
            if nvb_target > nvb:
                nvb_seed_added = nvb_target - nvb
                nvb = nvb_target
            bulk_first_gas_seed_done = True
            bulk_first_gas_seed_applied = True

        if (
            USE_FIRST_GAS_PRESSURE_SEED
            and SEED_DISLOCATION_ON_FIRST_GAS
            and (not dislocation_first_gas_seed_done)
            and Nd_old > 0.0
            and md_old <= 0.0
            and md_new > 0.0
        ):
            nvd_target, _Rd_seed_from_gas = vacancies_for_pressure_factor_from_gas(
                p,
                md_new,
                Nd_old,
                FIRST_GAS_PRESSURE_FACTOR,
            )
            if nvd_target > nvd:
                nvd_seed_added = nvd_target - nvd
                nvd = nvd_target
            dislocation_first_gas_seed_done = True
            dislocation_first_gas_seed_applied = True

        if p.update_bulk_vacancies and not bulk_first_gas_seed_applied:
            nvb, dnvb_dt = vacancy_concentration_implicit_step(p, Dv, R_b, N_b, mb_new, nvb, dt)
        else:
            dnvb_dt = (nvb - nvb_before_vacancy_update) / dt

        if not dislocation_first_gas_seed_applied:
            nvd, dnvd_dt = vacancy_concentration_implicit_step(p, Dv_d, R_d, Nd_old, md_new, nvd, dt)
        else:
            dnvd_dt = (nvd - nvd_before_vacancy_update) / dt

        if N_b > 0.0:
            V_b_growth = V_b + dt * (p.omega_fg / N_b * dmb_dt + omega_matrix(p) / N_b * dnvb_dt)
            V_b_growth = max(V_b_growth, p.min_volume)
        else:
            V_b_growth = 0.0

        if Nd_old > 0.0:
            dVd_growth_dt = p.omega_fg / Nd_old * dmd_dt + omega_matrix(p) / Nd_old * dnvd_dt
            V_d_growth = max(V_d + dt * dVd_growth_dt, p.min_volume)
        else:
            dVd_growth_dt = 0.0
            V_d_growth = 0.0

        lambda_d = coalescence_lambda(Vd_old, Nd_old)
        dVd_positive = max(V_d_growth - Vd_old, 0.0)
        if dVd_positive > 0.0 and Nd_old > 0.0:
            denominator = 1.0 + p.coalescence_d_scale * 4.0 * lambda_d * Nd_old * dVd_positive
            N_d = Nd_old / denominator
        else:
            N_d = Nd_old
        N_d = max(N_d, p.min_number_density)

        V_b = (p.omega_fg * max(mb_new, 0.0) + omega_matrix(p) * nvb) / N_b if N_b > 0.0 else 0.0
        V_d = (p.omega_fg * max(md_new, 0.0) + omega_matrix(p) * nvd) / N_d if N_d > 0.0 else 0.0
        V_b = max(V_b, p.min_volume)
        V_d = max(V_d, p.min_volume)
        R_b = radius_from_volume(V_b)
        R_d = radius_from_volume(V_d)

        delta_Rd_cap = max(R_d - Rd_old, 0.0)
        delta_Vcap = 4.0 * math.pi * (Rd_old + Rb_old) ** 2 * delta_Rd_cap
        if USE_BULK_DISLOCATION_CAPTURE:
            cap_raw_step = p.capture_scale * N_d * delta_Vcap
            f_cap = max(0.0, min(cap_raw_step, 1.0))
        else:
            cap_raw_step = 0.0
            f_cap = 0.0

        capture_raw_sum += cap_raw_step
        capture_fraction_sum += f_cap
        max_f_cap_step = max(max_f_cap_step, f_cap)

        if f_cap > 0.0 and N_b > 0.0:
            mb_before = max(mb_new, 0.0)
            nvb_before = max(nvb, 0.0)
            captured_bubbles = f_cap * N_b

            mb_new = (1.0 - f_cap) * mb_before
            md_new = max(md_new, 0.0) + f_cap * mb_before
            nvb = (1.0 - f_cap) * nvb_before
            nvd = max(nvd, 0.0) + f_cap * nvb_before
            N_b = (1.0 - f_cap) * N_b

            capture_bubbles_cumulative += captured_bubbles

            modes_c, modes_mb, modes_md = reset_modes_to_averages(c_new, mb_new, md_new, p.n_modes)

            V_b = (p.omega_fg * max(mb_new, 0.0) + omega_matrix(p) * nvb) / N_b if N_b > 0.0 else 0.0
            V_d = (p.omega_fg * max(md_new, 0.0) + omega_matrix(p) * nvd) / N_d if N_d > 0.0 else 0.0
            V_b = max(V_b, p.min_volume)
            V_d = max(V_d, p.min_volume)
            R_b = radius_from_volume(V_b)
            R_d = radius_from_volume(V_d)

        generated += beta * dt
        retained = max(c_new, 0.0) + max(mb_new, 0.0) + max(md_new, 0.0)
        q_gb = max(initial_gas + generated - retained, 0.0)
        t += dt

        last_rates = {
            "Dg": Dg, "Dv": Dv, "Dv_dislocation": Dv_d, "beta": beta,
            "g_b": g_b, "g_d": g_d,
            "b_b": b_b, "b_d": b_d,
            "b_b_gas": b_b_gas, "b_d_gas": b_d_gas,
            "nu_b": nu_b, "phi_b": phi_b, "phi_d": phi_d,
            "lambda_d": lambda_d,
            "dVd_growth_dt": dVd_growth_dt,
            "dnvb_dt": dnvb_dt, "dnvd_dt": dnvd_dt,
            "nvb_seed_added": locals().get("nvb_seed_added", 0.0),
            "nvd_seed_added": locals().get("nvd_seed_added", 0.0),
            "bulk_first_gas_seed_applied": float(locals().get("bulk_first_gas_seed_applied", False)),
            "dislocation_first_gas_seed_applied": float(locals().get("dislocation_first_gas_seed_applied", False)),
            "first_gas_pressure_factor": FIRST_GAS_PRESSURE_FACTOR,
            "seed_pressure_factor": SEED_PRESSURE_FACTOR,
            "seed_gas_atoms_per_bubble": SEED_GAS_ATOMS_PER_BUBBLE,
            "f_cap_step": f_cap,
            "cap_raw_step": cap_raw_step,
            "capture_fraction_sum": capture_fraction_sum,
            "capture_raw_sum": capture_raw_sum,
            "capture_bubbles_cumulative": capture_bubbles_cumulative,
            "max_f_cap_step": max_f_cap_step,
            **D_parts, **Dv_parts, **trapping_parts,
        }

        append_state(nu_b=nu_b, phi_b=phi_b, phi_d=phi_d, lambda_d=lambda_d, fcap=f_cap, capraw=cap_raw_step)

    if not keep_history:
        c_av = reconstruct_average(modes_c)
        mb_av = reconstruct_average(modes_mb)
        md_av = reconstruct_average(modes_md)
        p_b = pressure_internal(p, mb_av, nvb)
        p_d = pressure_internal(p, md_av, nvd)
        p_b_eq = pressure_equilibrium(p, R_b)
        p_d_eq = pressure_equilibrium(p, R_d)
        hist = {
            "time": [t],
            "burnup_percent_fima": [time_to_burnup_percent(t, p.fission_rate, p.lattice_parameter)],
            "c": [c_av], "mb": [mb_av], "md": [md_av],
            "Nb": [N_b], "Nd": [N_d],
            "Vb": [V_b], "Vd": [V_d],
            "Rb": [R_b], "Rd": [R_d],
            "nvb": [nvb], "nvd": [nvd],
            "generated": [generated],
            "retained": [retained],
            "q_gb": [q_gb],
            "swelling_b": [N_b * V_b],
            "swelling_d": [N_d * V_d],
            "swelling_ig": [N_b * V_b + N_d * V_d],
            "p_b": [p_b], "p_d": [p_d],
            "p_b_eq": [p_b_eq], "p_d_eq": [p_d_eq],
            "lambda_d": [last_rates.get("lambda_d", 0.0)],
            "nu_b": [last_rates.get("nu_b", 0.0)],
            "phi_b": [last_rates.get("phi_b", 0.0)],
            "phi_d": [last_rates.get("phi_d", 0.0)],
            "f_cap_step": [last_rates.get("f_cap_step", 0.0)],
            "cap_raw_step": [last_rates.get("cap_raw_step", 0.0)],
            "capture_fraction_sum": [capture_fraction_sum],
            "capture_raw_sum": [capture_raw_sum],
            "capture_bubbles_cumulative": [capture_bubbles_cumulative],
            "max_f_cap_step": [max_f_cap_step],
            "matrix_gas_percent": [100.0 * c_av / generated if generated > 0.0 else 0.0],
            "bulk_gas_percent": [100.0 * mb_av / generated if generated > 0.0 else 0.0],
            "dislocation_gas_percent": [100.0 * md_av / generated if generated > 0.0 else 0.0],
            "qgb_gas_percent": [100.0 * q_gb / generated if generated > 0.0 else 0.0],
        }

    return hist, last_rates


## Manual runner and plots

In [130]:

# ============================================================
# MANUAL RUNNER, DIAGNOSTICS, AND PLOTS
# ============================================================

from pathlib import Path
import csv
import math
from dataclasses import replace
from typing import List, Dict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

RUN_CACHE = {}

def ensure_output_dir(path=OUTPUT_DIR):
    Path(path).mkdir(parents=True, exist_ok=True)

def make_candidate(label=CASE_LABEL) -> Candidate:
    return Candidate(label=label, **dict(MANUAL_PARAMS))

# ---- Optional rho_d models ----
RHO_FAB_DEFAULT = 3.0e13
C1_RB = 1.6e14
F0_RB = 2.4
TREF_RB = 1025.0
RHO_SAT_RHO940 = 6.3571
RHO_SAT_RHOINF = 9.1036
RHO_SAT_TAU_K = 203.76

# Rizk 2023 empirical dislocation-density model.
# Eq. form: rho = rho0 * exp(c*F / max(Td, T0 - T)), capped at rho_cap.
# F must be the FIMA fraction, not the percent value.
def rho_rizk2023(T: float, burnup_percent: float) -> float:
    F_fima_fraction = float(burnup_percent) / 100.0
    denominator_K = max(float(RIZK2023_TD), float(RIZK2023_T0) - float(T))
    exponent = float(RIZK2023_C) * F_fima_fraction / denominator_K
    rho = float(RIZK2023_RHO0) * math.exp(exponent)
    return min(rho, float(RIZK2023_RHO_CAP))

def rho_sat_shape10(T: float) -> float:
    return RHO_SAT_RHOINF - (RHO_SAT_RHOINF - RHO_SAT_RHO940) * math.exp(-(float(T) - 940.0) / RHO_SAT_TAU_K)

def rho_sat_factor_raw(T: float) -> float:
    return rho_sat_shape10(float(T)) / rho_sat_shape10(TREF_RB)

def rho_burnup_1025(F_a_o: float) -> float:
    rho_bu = C1_RB * max(float(F_a_o) - F0_RB, 0.0)
    return max(RHO_FAB, rho_bu)

def effective_rho_d(T: float, burnup: float, cand: Candidate) -> float:
    if RHO_MODE == "constant":
        return cand.rho_d
    if RHO_MODE == "rhoSat_RayBlank":
        return max(rho_burnup_1025(burnup) * RHO_SCALE * rho_sat_factor_raw(T), 1.0e10)
    if RHO_MODE == "rizk2023":
        return max(rho_rizk2023(T, burnup), 1.0e10)
    raise ValueError(f"Unknown RHO_MODE={RHO_MODE!r}")

def run_model_point(T: float, burnup: float, cand: Candidate, dt_h=DT_H, n_modes=N_MODES, keep_history=False):
    rho_eff = effective_rho_d(T, burnup, cand)
    cache_key = (
        cand, float(T), float(burnup), float(dt_h), int(n_modes), bool(keep_history),
        USE_PHI_GAS_RESOLUTION, USE_NUCLEATION_MASS_COUPLING, USE_BULK_DISLOCATION_CAPTURE,
        A20_VU_DEFAULT, RHO_MODE, RHO_SCALE, XE_DIFFUSIVITY_MODE, VU_DIFFUSIVITY_MODE,
    )
    if not keep_history and cache_key in RUN_CACHE:
        return RUN_CACHE[cache_key]

    p = UNParameters(
        temperature=float(T),
        fission_rate=cand.fission_rate,
        grain_radius=GRAIN_RADIUS,
        target_burnup_percent_fima=float(burnup),
        dt=float(dt_h) * 3600.0,
        n_modes=int(n_modes),
        xe_yield=XE_YIELD,

        f_n=cand.f_n,
        K_d=cand.K_d,
        rho_d=rho_eff,

        Dv_scale=cand.Dv_scale,
        Dv_D1_scale=cand.Dv_D1_scale,
        Dv_D2_scale=cand.Dv_D2_scale,
        Dv_dislocation_scale=cand.Dv_dislocation_scale,
        vacancy_diffusivity_mode=VU_DIFFUSIVITY_MODE,
        A20_vU=A20_VU_DEFAULT,
        B21_vU=B21_VU,
        B22_vU=B22_VU,

        Dg_scale=cand.Dg_scale,
        Dg_D1_scale=cand.Dg_D1_scale,
        Dg_D3_scale=cand.Dg_D3_scale,
        Dg_dislocation_scale=cand.Dg_dislocation_scale,
        D2_xe_scale=getattr(cand, "D2_xe_scale", 1.0),
        xe_diffusivity_mode=XE_DIFFUSIVITY_MODE,

        b_scale=cand.b_scale,
        b_bulk_scale=cand.b_bulk_scale,
        b_dislocation_scale=cand.b_dislocation_scale,

        gb_scale=cand.gb_scale,
        gd_scale=cand.gd_scale,
        gd_bubble_scale=cand.gd_bubble_scale,
        gd_line_scale=cand.gd_line_scale,
        gd_line_alpha=cand.gd_line_alpha,

        coalescence_d_scale=cand.coalescence_d_scale,
        capture_scale=cand.capture_scale,
    )

    hist, rates = solve_UN(p, keep_history=keep_history)

    # Last values.
    out = {
        "label": cand.label,
        "xe_diffusivity_mode": XE_DIFFUSIVITY_MODE,
        "vacancy_diffusivity_mode": VU_DIFFUSIVITY_MODE,
        "T": float(T),
        "burnup": float(burnup),
        "rho_d_eff": rho_eff,

        "c": hist["c"][-1],
        "mb": hist["mb"][-1],
        "md": hist["md"][-1],
        "Nb": hist["Nb"][-1],
        "Nd": hist["Nd"][-1],
        "Rb_nm": hist["Rb"][-1] * 1e9,
        "Rd_nm": hist["Rd"][-1] * 1e9,

        "swelling_b_percent": 100.0 * hist["swelling_b"][-1],
        "swelling_d_percent": 100.0 * hist["swelling_d"][-1],
        "swelling_ig_percent": 100.0 * hist["swelling_ig"][-1],

        "p_b": hist["p_b"][-1],
        "p_b_eq": hist["p_b_eq"][-1],
        "p_d": hist["p_d"][-1],
        "p_d_eq": hist["p_d_eq"][-1],
        "p_b_over_eq": hist["p_b"][-1] / hist["p_b_eq"][-1] if hist["p_b_eq"][-1] else math.nan,
        "p_d_over_eq": hist["p_d"][-1] / hist["p_d_eq"][-1] if hist["p_d_eq"][-1] else math.nan,

        "matrix_gas_percent": hist["matrix_gas_percent"][-1],
        "bulk_gas_percent": hist["bulk_gas_percent"][-1],
        "dislocation_gas_percent": hist["dislocation_gas_percent"][-1],
        "qgb_gas_percent": hist["qgb_gas_percent"][-1],

        "q_gb": hist["q_gb"][-1],
        "generated": hist["generated"][-1],
        "retained": hist["retained"][-1],

        "lambda_d": hist["lambda_d"][-1],
        "nu_b": hist["nu_b"][-1],
        "phi_b": hist["phi_b"][-1],
        "phi_d": hist["phi_d"][-1],
        "f_cap_step": hist["f_cap_step"][-1],
        "cap_raw_step": hist["cap_raw_step"][-1],
        "capture_fraction_sum": hist["capture_fraction_sum"][-1],
        "capture_raw_sum": hist["capture_raw_sum"][-1],
        "capture_bubbles_cumulative": hist["capture_bubbles_cumulative"][-1],
        "max_f_cap_step": hist["max_f_cap_step"][-1],

        "hist": hist if keep_history else None,
        "rates": rates,
    }

    # Add rate diagnostics.
    for k, v in rates.items():
        if isinstance(v, (int, float)):
            out[k] = v

    # Geometry diagnostics.
    Rb = hist["Rb"][-1]
    Rd = hist["Rd"][-1]
    Nb = hist["Nb"][-1]
    Nd = hist["Nd"][-1]
    out["psi_b"] = Rb / wigner_seitz_delta(Nb) if Nb > 0 and math.isfinite(Rb) else math.nan
    out["psi_d"] = Rd / wigner_seitz_delta(Nd) if Nd > 0 and math.isfinite(Rd) else math.nan
    out["porosity_b"] = hist["swelling_b"][-1]
    out["porosity_d"] = hist["swelling_d"][-1]
    out["valid_single_size"] = (
        math.isfinite(out["Rd_nm"])
        and math.isfinite(out["Nd"])
        and (out["Nd"] > 1.0e18)
        and (out["psi_d"] < 0.8 if math.isfinite(out["psi_d"]) else False)
        and (out["porosity_d"] < 0.5 if math.isfinite(out["porosity_d"]) else False)
    )

    if not keep_history:
        RUN_CACHE[cache_key] = out
    return out

def temperature_grid(tmax):
    n = int(round((float(tmax) - T_MIN) / T_STEP))
    return [T_MIN + i * T_STEP for i in range(n + 1)]

def simulate_grid(cand: Candidate):
    RUN_CACHE.clear()
    rows = []
    for bu in BURNUPS:
        Ts = temperature_grid(T_MAX_DIAGNOSTIC)
        print(f"Running burnup {bu:g}% FIMA: {len(Ts)} temperature points")
        for T in Ts:
            rows.append(run_model_point(T, bu, cand, DT_H, N_MODES, keep_history=False))
    return rows

def rows_for_burnup(rows, bu):
    return sorted([r for r in rows if abs(r["burnup"] - float(bu)) < 1e-9], key=lambda r: r["T"])

def row_near(rows, bu, T):
    sub = rows_for_burnup(rows, bu)
    return min(sub, key=lambda r: abs(r["T"] - T))

def savefig(name):
    ensure_output_dir()
    path = Path(OUTPUT_DIR) / name
    plt.savefig(path, dpi=180, bbox_inches="tight")
    if not SHOW_PLOTS:
        plt.close()
    return path

def write_csv(rows, cand):
    ensure_output_dir()
    path = Path(OUTPUT_DIR) / f"{cand.label}_grid.csv"
    df = pd.DataFrame([{k: v for k, v in r.items() if k not in ("hist", "rates")} for r in rows])
    df.to_csv(path, index=False)
    return path, df

# ---- plotting helpers ----

def plot_exp_swelling_for_burnup(bu):
    series_names = sorted({p["series"] for p in EXP_SWELLING_T if abs(p["burnup"] - bu) < 1e-9})
    for series in series_names:
        pts = [p for p in EXP_SWELLING_T if abs(p["burnup"] - bu) < 1e-9 and p["series"] == series]
        marker = "x" if "119" in series else "^"
        plt.scatter([p["T"] for p in pts], [p["swelling"] for p in pts], marker=marker, s=65, label=f"Exp P2 {series}", zorder=5)

def plot_swelling_T(rows, bu, cand):
    sub = [r for r in rows_for_burnup(rows, bu) if r["T"] <= T_MAX_SWELLING]
    Ts = [r["T"] for r in sub]
    plt.figure(figsize=(9, 5.8))
    plt.plot(Ts, [r["swelling_d_percent"] for r in sub], label="Model dislocation/P2 swelling")
    plt.plot(Ts, [r["swelling_b_percent"] for r in sub], "--", label="Model bulk swelling")
    plot_exp_swelling_for_burnup(bu)
    plt.xlabel("Temperature [K]")
    plt.ylabel("Fission gas swelling [%]")
    plt.title(f"{cand.label}: swelling vs T at {bu:.1f}% FIMA")
    plt.grid(True, alpha=0.3)
    plt.legend(fontsize=8)
    return savefig(f"{cand.label}_swelling_T_{bu:.1f}FIMA.png")

def plot_swelling_burnup_1600(cand):
    burnup_grid = [0.2, 0.5, 0.8, 1.1, 1.3, 1.6, 2.0, 2.5, 3.2, 4.0, 5.0, 6.0]
    rows_bu = [run_model_point(1600.0, bu, cand, DT_H, N_MODES, keep_history=False) for bu in burnup_grid]
    plt.figure(figsize=(9, 5.8))
    plt.plot(burnup_grid, [r["swelling_d_percent"] for r in rows_bu], label="Model dislocation/P2 swelling")
    plt.plot(burnup_grid, [r["swelling_b_percent"] for r in rows_bu], "--", label="Model bulk swelling")
    plt.scatter([p["burnup"] for p in EXP_SWELLING_BURNUP_1600], [p["swelling"] for p in EXP_SWELLING_BURNUP_1600], marker="^", s=80, label="Exp P2 approx")
    plt.xlabel("Burnup [% FIMA]")
    plt.ylabel("Fission gas swelling [%]")
    plt.title(f"{cand.label}: swelling vs burnup at 1600 K")
    plt.grid(True, alpha=0.3)
    plt.legend(fontsize=8)
    return savefig(f"{cand.label}_swelling_vs_burnup_1600K.png")

def existing_bulk_value(r, key):
    """Return finite bulk value only when the bulk population exists."""
    if r.get("Nb", 0.0) > 0.0 and r.get("Rb_nm", 0.0) > 0.0 and r.get("mb", 0.0) > 0.0:
        val = r.get(key, math.nan)
        return val if math.isfinite(val) and val > 0.0 else math.nan
    return math.nan

def existing_dislocation_value(r, key):
    """Return finite dislocation value only when the dislocation population has gas/volume."""
    if r.get("Nd", 0.0) > 0.0 and r.get("Rd_nm", 0.0) > 0.0 and r.get("md", 0.0) > 0.0:
        val = r.get(key, math.nan)
        return val if math.isfinite(val) and val > 0.0 else math.nan
    return math.nan

def plot_radius_concentration_T(rows, cand, bu=1.3):
    """Plot bulk and dislocation R/N together, so bulk disappearance is visible."""
    sub = rows_for_burnup(rows, bu)
    Ts = [r["T"] for r in sub]
    fig, axes = plt.subplots(1, 2, figsize=(13, 4.8))

    ax = axes[0]
    ax.semilogy(Ts, [existing_dislocation_value(r, "Rd_nm") for r in sub], label="Model R_d dislocation")
    ax.semilogy(Ts, [existing_bulk_value(r, "Rb_nm") for r in sub], "--", label="Model R_b bulk")
    if abs(bu - 1.3) < 1e-9:
        ax.scatter([p["T"] for p in EXP_RD_T_13], [p["R_nm"] for p in EXP_RD_T_13], marker="^", color="red", label="Rizk Fig. 7/8 exp")
    ax.set_xlabel("Temperature [K]")
    ax.set_ylabel("Bubble radius [nm]")
    ax.set_title(f"Bulk + dislocation bubble radius ({bu:.1f}% FIMA)")
    ax.grid(True, alpha=0.3, which="both")
    ax.legend(fontsize=8)

    ax = axes[1]
    ax.semilogy(Ts, [existing_dislocation_value(r, "Nd") for r in sub], label="Model N_d dislocation")
    ax.semilogy(Ts, [existing_bulk_value(r, "Nb") for r in sub], "--", label="Model N_b bulk")
    if abs(bu - 1.3) < 1e-9:
        ax.scatter([p["T"] for p in EXP_ND_T_13], [p["N"] for p in EXP_ND_T_13], marker="^", color="red", label="Rizk Fig. 8 exp")
    ax.set_xlabel("Temperature [K]")
    ax.set_ylabel("Bubble concentration [m$^{-3}$]")
    ax.set_title(f"Bulk + dislocation bubble concentration ({bu:.1f}% FIMA)")
    ax.grid(True, alpha=0.3, which="both")
    ax.legend(fontsize=8)

    plt.tight_layout()
    return savefig(f"{cand.label}_RbRd_NbNd_T_{bu:.1f}FIMA.png")

def plot_pressure_T(rows, cand, bu):
    sub = rows_for_burnup(rows, bu)
    Ts = [r["T"] for r in sub]
    plt.figure(figsize=(9, 5.8))
    # Mask non-existing populations; otherwise p_eq uses R_min=1e-15 and shows fake ~1e15 Pa plateaus.
    plt.semilogy(Ts, [existing_bulk_value(r, "p_b") for r in sub], label="p_b bulk")
    plt.semilogy(Ts, [existing_bulk_value(r, "p_b_eq") for r in sub], "--", label="p_b,eq bulk")
    plt.semilogy(Ts, [existing_dislocation_value(r, "p_d") for r in sub], label="p_d dislocation")
    plt.semilogy(Ts, [existing_dislocation_value(r, "p_d_eq") for r in sub], "--", label="p_d,eq dislocation")
    plt.xlabel("Temperature [K]")
    plt.ylabel("Pressure [Pa]")
    plt.title(f"{cand.label}: pressure and equilibrium pressure at {bu:.1f}% FIMA")
    plt.grid(True, alpha=0.3, which="both")
    plt.legend(fontsize=8)
    return savefig(f"{cand.label}_pressure_T_{bu:.1f}FIMA.png")

def plot_pressure_ratio_T(rows, cand, bu):
    sub = rows_for_burnup(rows, bu)
    Ts = [r["T"] for r in sub]
    plt.figure(figsize=(9, 5.8))
    bulk_ratio = [
        (r["p_b_over_eq"] if math.isfinite(r.get("p_b_over_eq", math.nan)) and not math.isnan(existing_bulk_value(r, "p_b")) else math.nan)
        for r in sub
    ]
    disl_ratio = [
        (r["p_d_over_eq"] if math.isfinite(r.get("p_d_over_eq", math.nan)) and not math.isnan(existing_dislocation_value(r, "p_d")) else math.nan)
        for r in sub
    ]
    plt.plot(Ts, bulk_ratio, label="p_b/p_b,eq")
    plt.plot(Ts, disl_ratio, label="p_d/p_d,eq")
    plt.axhline(1.0, color="black", linewidth=1, linestyle="--")
    plt.xlabel("Temperature [K]")
    plt.ylabel("Pressure ratio [-]")
    plt.title(f"{cand.label}: pressure ratio at {bu:.1f}% FIMA")
    plt.grid(True, alpha=0.3)
    plt.legend(fontsize=8)
    return savefig(f"{cand.label}_pressure_ratio_T_{bu:.1f}FIMA.png")

def plot_gas_partition(rows, cand, bu):
    sub = rows_for_burnup(rows, bu)
    Ts = [r["T"] for r in sub]
    y1 = [r["matrix_gas_percent"] for r in sub]
    y2 = [r["bulk_gas_percent"] for r in sub]
    y3 = [r["dislocation_gas_percent"] for r in sub]
    y4 = [r["qgb_gas_percent"] for r in sub]
    plt.figure(figsize=(9, 5.8))
    plt.stackplot(Ts, y1, y2, y3, y4, labels=["Matrix (solution)", "Bulk bubbles", "Dislocation bubbles", "q_gb (grain face)"], alpha=0.85)
    plt.xlabel("Temperature [K]")
    plt.ylabel("Fraction of generated gas [%]")
    plt.title(f"{cand.label}: gas partition vs T at {bu:.1f}% FIMA")
    plt.ylim(0, 100)
    plt.grid(True, alpha=0.25)
    plt.legend(fontsize=8, loc="center right")
    return savefig(f"{cand.label}_gas_partition_{bu:.1f}FIMA.png")

def plot_diffusivities(rows, cand):
    sub = rows_for_burnup(rows, BURNUPS[0])
    Ts = [r["T"] for r in sub]
    plt.figure(figsize=(9, 5.8))
    plt.semilogy(Ts, [max(r.get("Dg", math.nan), 1e-300) for r in sub], label="Dg = D1_Xe + D3_Xe")
    plt.semilogy(Ts, [max(r.get("Dv", math.nan), 1e-300) for r in sub], label="Dv total")
    plt.semilogy(Ts, [max(r.get("Dv1", math.nan), 1e-300) for r in sub], "--", label="Dv1 thermal")
    plt.semilogy(Ts, [max(r.get("Dv2", math.nan), 1e-300) for r in sub], ":", label="Dv2 irradiation")
    plt.xlabel("Temperature [K]")
    plt.ylabel("Diffusivity [m$^2$/s]")
    plt.title(f"{cand.label}: diffusivities vs T")
    plt.grid(True, alpha=0.3, which="both")
    plt.legend(fontsize=8)
    return savefig(f"{cand.label}_diffusivities_T.png")

def plot_capture_diagnostic(rows, cand, bu):
    sub = rows_for_burnup(rows, bu)
    Ts = [r["T"] for r in sub]
    plt.figure(figsize=(9, 5.8))
    plt.semilogy(Ts, [max(r["max_f_cap_step"], 1e-300) for r in sub], label="max f_cap step, clipped fraction")
    plt.semilogy(Ts, [max(r["capture_fraction_sum"], 1e-300) for r in sub], label="sum clipped f_cap steps, diagnostic")
    plt.semilogy(Ts, [max(r["capture_raw_sum"], 1e-300) for r in sub], label="sum raw capture hazard, diagnostic")
    plt.xlabel("Temperature [K]")
    plt.ylabel("Capture diagnostic")
    plt.title(f"{cand.label}: capture diagnostics at {bu:.1f}% FIMA")
    plt.grid(True, alpha=0.3, which="both")
    plt.legend(fontsize=8)
    return savefig(f"{cand.label}_capture_diagnostic_{bu:.1f}FIMA.png")

def plot_rho_surface(cand):
    # Simple visualization of active rho mode.
    from mpl_toolkits.mplot3d import Axes3D  # noqa: F401
    T_grid = np.arange(900, 2101, 50)
    F_grid = np.linspace(0, 100, 40)
    TT, FF = np.meshgrid(T_grid, F_grid)
    ZZ = np.vectorize(lambda T, F: math.log10(effective_rho_d(T, F/20.0 if False else F, cand)))(TT, FF)
    fig = plt.figure(figsize=(9, 6.2))
    ax = fig.add_subplot(111, projection="3d")
    surf = ax.plot_surface(TT, FF, ZZ, alpha=0.65)
    ax.set_xlabel("Temperature [K]")
    ax.set_ylabel("Burnup [MWd/kgHM proxy]")
    ax.set_zlabel(r"$\log_{10}\rho_d$ [m$^{-2}$]")
    ax.set_title(f"rho_d(F,T) -- {RHO_MODE}, rho_d={cand.rho_d:.2e} m^-2")
    return savefig(f"{cand.label}_rho_d_surface.png")

def make_all_plots(rows, cand):
    saved = []
    for bu in BURNUPS:
        saved.append(plot_swelling_T(rows, bu, cand))
    saved.append(plot_swelling_burnup_1600(cand))
    # R and N diagnostics now include both bulk and dislocation bubbles.
    for bu in BURNUPS:
        saved.append(plot_radius_concentration_T(rows, cand, bu))
    for bu in BURNUPS:
        saved.append(plot_pressure_T(rows, cand, bu))
        saved.append(plot_pressure_ratio_T(rows, cand, bu))
        saved.append(plot_gas_partition(rows, cand, bu))
        saved.append(plot_capture_diagnostic(rows, cand, bu))
    saved.append(plot_diffusivities(rows, cand))
    saved.append(plot_rho_surface(cand))
    return saved

def print_summary(rows, cand):
    print("\n" + "=" * 120)
    print(f"Manual case: {cand.label}")
    print("=" * 120)
    print("Physics switches:")
    print(f"  USE_PHI_GAS_RESOLUTION       = {USE_PHI_GAS_RESOLUTION}")
    print(f"  USE_NUCLEATION_MASS_COUPLING = {USE_NUCLEATION_MASS_COUPLING}")
    print(f"  USE_BULK_DISLOCATION_CAPTURE = {USE_BULK_DISLOCATION_CAPTURE}")
    print("Key parameters:")
    print(f"  K_d                          = {cand.K_d:.6g}")
    print(f"  rho_d                        = {cand.rho_d:.6g}")
    print(f"  N_d0 = K_d*rho_d             = {cand.K_d*cand.rho_d:.6e}")
    print(f"  A20_vU active                = {A20_VU_DEFAULT:.6e}")
    print(f"  Dv formula                   = sqrt(Fdot)*A20*exp(B21/kBT + B22/(kBT)^2)")
    print(f"  USE_FIRST_GAS_PRESSURE_SEED  = {USE_FIRST_GAS_PRESSURE_SEED}")
    print(f"  FIRST_GAS_PRESSURE_FACTOR    = {FIRST_GAS_PRESSURE_FACTOR:.6g}")
    print(f"  SEED_BULK_ON_FIRST_GAS       = {SEED_BULK_ON_FIRST_GAS}")
    print(f"  SEED_DISLOCATION_ON_FIRST_GAS= {SEED_DISLOCATION_ON_FIRST_GAS}")
    print(f"  legacy dimer seed at birth   = {USE_EQUILIBRIUM_DIMER_SEED}")
    print("\nSelected diagnostics:")
    header = f"{'bu [%]':>7s} {'T [K]':>7s} {'sw_d [%]':>10s} {'R_d [nm]':>10s} {'N_d [m^-3]':>13s} {'R_b [nm]':>10s} {'N_b [m^-3]':>13s} {'p_d/peq':>9s} {'qgb [%]':>9s}"
    print(header)
    print("-" * len(header))
    for bu in BURNUPS:
        for T in [1200.0, 1600.0, 1800.0, 2000.0]:
            if T < T_MIN or T > T_MAX_DIAGNOSTIC:
                continue
            r = row_near(rows, bu, T)
            rb = r['Rb_nm'] if r.get('Nb', 0.0) > 0.0 and r.get('Rb_nm', 0.0) > 0.0 else math.nan
            nb_ = r['Nb'] if r.get('Nb', 0.0) > 0.0 else math.nan
            print(f"{bu:7.2f} {r['T']:7.0f} {r['swelling_d_percent']:10.3g} {r['Rd_nm']:10.3g} {r['Nd']:13.3e} {rb:10.3g} {nb_:13.3e} {r['p_d_over_eq']:9.3g} {r['qgb_gas_percent']:9.3g}")
    print("=" * 120)

def main():
    ensure_output_dir()
    cand = make_candidate()
    rows = simulate_grid(cand)
    csv_path, df = write_csv(rows, cand)
    print_summary(rows, cand)
    saved = make_all_plots(rows, cand)
    print(f"\nCSV written to: {csv_path}")
    print("Plots written:")
    for p in saved:
        print(f"  - {p}")
    return rows, df, saved

def inspect_point(T=1600.0, burnup=1.3, keep_history=True):
    cand = make_candidate()
    out = run_model_point(T, burnup, cand, DT_H, N_MODES, keep_history=keep_history)
    keys = ["T", "burnup", "swelling_d_percent", "swelling_b_percent", "Rd_nm", "Nd", "p_d", "p_d_eq", "p_d_over_eq", "qgb_gas_percent", "phi_b", "phi_d", "max_f_cap_step"]
    print({k: out.get(k) for k in keys})
    print("\nrates:")
    for k in ["xe_diffusivity_mode", "vacancy_diffusivity_mode", "Dg", "Dg_dislocation", "Dv", "Dv_dislocation", "Dv1", "Dv2", "A20_vU_active", "g_b", "g_d", "b_b", "b_d", "b_b_gas", "b_d_gas", "term_bubbles", "term_dislocation"]:
        print(f"  {k:20s} = {out.get(k)}")
    return out


## Run

In [131]:

# Run the current manual configuration
rows, df, saved_plots = main()


Running burnup 1.1% FIMA: 23 temperature points
Running burnup 1.3% FIMA: 23 temperature points
Running burnup 3.2% FIMA: 23 temperature points

Manual case: 4test_UN
Physics switches:
  USE_PHI_GAS_RESOLUTION       = False
  USE_NUCLEATION_MASS_COUPLING = True
  USE_BULK_DISLOCATION_CAPTURE = True
Key parameters:
  K_d                          = 500000
  rho_d                        = 3e+13
  N_d0 = K_d*rho_d             = 1.500000e+19
  A20_vU active                = 4.630452e-29
  Dv formula                   = sqrt(Fdot)*A20*exp(B21/kBT + B22/(kBT)^2)
  USE_FIRST_GAS_PRESSURE_SEED  = True
  FIRST_GAS_PRESSURE_FACTOR    = 0.333333
  SEED_BULK_ON_FIRST_GAS       = True
  SEED_DISLOCATION_ON_FIRST_GAS= True
  legacy dimer seed at birth   = False

Selected diagnostics:
 bu [%]   T [K]   sw_d [%]   R_d [nm]    N_d [m^-3]   R_b [nm]    N_b [m^-3]   p_d/peq   qgb [%]
------------------------------------------------------------------------------------------------
   1.10    1200       1.65

## Optional single-point inspection

In [132]:

# Optional quick single-point inspection
# Change T_INSPECT / BU_INSPECT and run this cell.
T_INSPECT = 1600.0
BU_INSPECT = 1.3
point = inspect_point(T_INSPECT, BU_INSPECT, keep_history=True)


{'T': 1600.0, 'burnup': 1.3, 'swelling_d_percent': 7.522306075789464, 'swelling_b_percent': 0.4041653128435694, 'Rd_nm': 118.5732480738085, 'Nd': 1.0772146254792305e+19, 'p_d': 20314857.90404882, 'p_d_eq': 18722604.264143232, 'p_d_over_eq': 1.0850444530815089, 'qgb_gas_percent': 11.015015976890092, 'phi_b': 0.00028745129747347845, 'phi_d': 1.6806116423317522e-07, 'max_f_cap_step': 0.0006836278154689692}

rates:
  xe_diffusivity_mode  = rizk2025_refit_plot
  vacancy_diffusivity_mode = rizk2025_refit_full
  Dg                   = 1.1650920113912498e-18
  Dg_dislocation       = 1.1650920113912497e-17
  Dv                   = 2.783735691844339e-20
  Dv_dislocation       = 2.783735691844339e-19
  Dv1                  = 2.6363059898109155e-20
  Dv2                  = 1.4742970203342355e-21
  A20_vU_active        = nan
  g_b                  = 0.0006362916737484829
  g_d                  = 0.0005949731811097548
  b_b                  = 8.314206045408959e-06
  b_d                  = 3.83088492


## Guard diagnostics (post-processing only)

Queste celle **non modificano il solver**. Usano `rows`, `df` e le funzioni già definite sopra per aggiungere grafici diagnostici sui guard numerici:

- `max_f_cap_step = 1` → clipping della bulk→dislocation capture;
- `psi_b`, `psi_d` → controllo `psi >= 1`, cioè bolla grande quanto/più della cella di Wigner-Seitz;
- `xi_b`, `xi_d` → porosità della popolazione, con `xi = V N = psi^3`;
- `lambda_d` → fattore di coalescenza, che esplode quando `xi -> 1`;
- pressure floor → falsi `p_eq ~ 1e15 Pa` quando il raggio è zero/non fisico;
- `dnvb_dt`, `dnvd_dt` → assorbimento di vacanze, per capire se la crescita di volume sta partendo da lì.

Sono diagnostiche **finali per ogni temperatura**. Per vedere la storia temporale interna di un punto specifico, usa la cella `plot_guard_history(...)` in fondo.


In [133]:

# ============================================================
# POST-PROCESSING GUARD DIAGNOSTICS
# This cell does NOT modify the solver. It only analyses rows/hist.
# ============================================================

import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

def _finite_positive(x):
    try:
        return math.isfinite(float(x)) and float(x) > 0.0
    except Exception:
        return False

def _safe_float(x, default=math.nan):
    try:
        x = float(x)
        return x if math.isfinite(x) else default
    except Exception:
        return default

def _xi_from_row(r, pop):
    # In this notebook porosity_b/d is exactly V*N for each population.
    key = "porosity_b" if pop == "b" else "porosity_d"
    return _safe_float(r.get(key, math.nan))

def _psi_from_row(r, pop):
    key = "psi_b" if pop == "b" else "psi_d"
    return _safe_float(r.get(key, math.nan))

def _lambda_clip_value():
    # coalescence_lambda clips xi to 0.999999.
    xi = 0.999999
    return (2.0 - xi) / (2.0 * (1.0 - xi) ** 3)

def _zeta_from_psi_for_plot(psi):
    # Original analytic expression, with NaN outside the physical domain psi < 1.
    # This is only for plotting; it does not touch solver behavior.
    psi = np.asarray(psi, dtype=float)
    z = np.full_like(psi, np.nan, dtype=float)
    mask = np.isfinite(psi) & (psi > 0.0) & (psi < 1.0)
    den = -psi[mask]**6 + 5.0 * psi[mask]**2 - 9.0 * psi[mask] + 5.0
    good = den > 0.0
    vals = np.full(mask.sum(), np.nan)
    vals[good] = 10.0 * psi[mask][good] * (1.0 + psi[mask][good]**3) / den[good]
    z[mask] = vals
    return z

def guard_dataframe_from_rows(rows):
    """Build a dataframe with final-state guard indicators for each T/burnup point."""
    records = []
    lam_clip = _lambda_clip_value()
    for r in rows:
        psi_b = _psi_from_row(r, "b")
        psi_d = _psi_from_row(r, "d")
        xi_b = _xi_from_row(r, "b")
        xi_d = _xi_from_row(r, "d")
        lam = _safe_float(r.get("lambda_d", math.nan))

        fcap_max = _safe_float(r.get("max_f_cap_step", math.nan))
        cap_raw_sum = _safe_float(r.get("capture_raw_sum", math.nan))
        cap_clip_sum = _safe_float(r.get("capture_fraction_sum", math.nan))

        p_b_eq = _safe_float(r.get("p_b_eq", math.nan))
        p_d_eq = _safe_float(r.get("p_d_eq", math.nan))
        rb_nm = _safe_float(r.get("Rb_nm", math.nan))
        rd_nm = _safe_float(r.get("Rd_nm", math.nan))
        nb = _safe_float(r.get("Nb", math.nan))
        nd = _safe_float(r.get("Nd", math.nan))

        rec = dict(r)
        rec.update({
            "xi_b": xi_b,
            "xi_d": xi_d,
            "zeta_b_final": float(_zeta_from_psi_for_plot([psi_b])[0]) if math.isfinite(psi_b) else math.nan,
            "zeta_d_final": float(_zeta_from_psi_for_plot([psi_d])[0]) if math.isfinite(psi_d) else math.nan,

            # Guard/invalid indicators inferred from final values.
            "guard_fcap_clipped": bool(math.isfinite(fcap_max) and fcap_max >= 1.0 - 1e-12),
            "guard_capture_raw_exceeded_clipped_sum": bool(
                math.isfinite(cap_raw_sum) and math.isfinite(cap_clip_sum) and cap_raw_sum > cap_clip_sum + 1e-12
            ),
            "guard_psi_b_ge_1": bool(math.isfinite(psi_b) and psi_b >= 1.0),
            "guard_psi_d_ge_1": bool(math.isfinite(psi_d) and psi_d >= 1.0),
            "guard_xi_b_ge_1": bool(math.isfinite(xi_b) and xi_b >= 1.0),
            "guard_xi_d_ge_1": bool(math.isfinite(xi_d) and xi_d >= 1.0),
            "near_xi_b_08": bool(math.isfinite(xi_b) and xi_b >= 0.8),
            "near_xi_d_08": bool(math.isfinite(xi_d) and xi_d >= 0.8),
            "near_psi_b_08": bool(math.isfinite(psi_b) and psi_b >= 0.8),
            "near_psi_d_08": bool(math.isfinite(psi_d) and psi_d >= 0.8),
            "guard_lambda_near_clip": bool(math.isfinite(lam) and lam >= 0.01 * lam_clip),
            "guard_pressure_floor_b": bool(math.isfinite(p_b_eq) and p_b_eq > 1e14),
            "guard_pressure_floor_d": bool(math.isfinite(p_d_eq) and p_d_eq > 1e14),
            "guard_nonfinite_R_or_N": bool(
                not math.isfinite(rb_nm) or not math.isfinite(rd_nm) or not math.isfinite(nb) or not math.isfinite(nd)
            ),
            "invalid_single_size_final": bool(not r.get("valid_single_size", False)),
        })
        records.append(rec)
    return pd.DataFrame(records)

def _save_guard_plot(name):
    ensure_output_dir()
    path = Path(OUTPUT_DIR) / name
    plt.savefig(path, dpi=180, bbox_inches="tight")
    if not SHOW_PLOTS:
        plt.close()
    return path

def plot_guard_geometry_summary(rows, cand=None):
    """For each burnup, plot psi, xi, lambda and zeta final values vs T."""
    gdf = guard_dataframe_from_rows(rows)
    label = cand.label if cand is not None else CASE_LABEL
    saved = []

    for bu in BURNUPS:
        sub = gdf[np.isclose(gdf["burnup"], float(bu))].sort_values("T")
        if sub.empty:
            continue
        Ts = sub["T"].to_numpy()

        fig, axes = plt.subplots(2, 2, figsize=(13, 8.5))

        ax = axes[0, 0]
        ax.plot(Ts, sub["psi_b"], "--", label=r"$\psi_b=R_b/\delta_b$")
        ax.plot(Ts, sub["psi_d"], label=r"$\psi_d=R_d/\delta_d$")
        ax.axhline(0.8, linestyle=":", linewidth=1, label=r"$\psi=0.8$")
        ax.axhline(1.0, linestyle="--", linewidth=1, label=r"$\psi=1$ guard")
        ax.set_ylabel(r"$\psi$ [-]")
        ax.set_title(f"{label}: psi at {bu:.1f}% FIMA")
        ax.grid(True, alpha=0.3)
        ax.legend(fontsize=8)

        ax = axes[0, 1]
        ax.plot(Ts, sub["xi_b"], "--", label=r"$\xi_b=V_bN_b$")
        ax.plot(Ts, sub["xi_d"], label=r"$\xi_d=V_dN_d$")
        ax.axhline(0.5, linestyle=":", linewidth=1, label=r"$\xi=0.5$")
        ax.axhline(1.0, linestyle="--", linewidth=1, label=r"$\xi=1$ invalid")
        ax.set_ylabel(r"$\xi$ [-]")
        ax.set_title(f"{label}: porosity/volume fraction at {bu:.1f}% FIMA")
        ax.grid(True, alpha=0.3)
        ax.legend(fontsize=8)

        ax = axes[1, 0]
        ax.semilogy(Ts, np.maximum(sub["lambda_d"].to_numpy(dtype=float), 1e-300), label=r"$\lambda_d$")
        ax.axhline(_lambda_clip_value(), linestyle="--", linewidth=1, label=r"$\lambda(\xi=0.999999)$")
        ax.set_xlabel("Temperature [K]")
        ax.set_ylabel(r"$\lambda_d$ [-]")
        ax.set_title(f"{label}: coalescence factor at {bu:.1f}% FIMA")
        ax.grid(True, which="both", alpha=0.3)
        ax.legend(fontsize=8)

        ax = axes[1, 1]
        ax.semilogy(Ts, np.maximum(sub["zeta_b_final"].to_numpy(dtype=float), 1e-300), "--", label=r"$\zeta_b$ final, only if $\psi<1$")
        ax.semilogy(Ts, np.maximum(sub["zeta_d_final"].to_numpy(dtype=float), 1e-300), label=r"$\zeta_d$ final, only if $\psi<1$")
        ax.set_xlabel("Temperature [K]")
        ax.set_ylabel(r"$\zeta$ [-]")
        ax.set_title(f"{label}: zeta geometry factor at {bu:.1f}% FIMA")
        ax.grid(True, which="both", alpha=0.3)
        ax.legend(fontsize=8)

        plt.tight_layout()
        saved.append(_save_guard_plot(f"{label}_guard_geometry_{bu:.1f}FIMA.png"))

    return saved

def plot_guard_activation_summary(rows, cand=None):
    """For each burnup, plot guard indicators vs T."""
    gdf = guard_dataframe_from_rows(rows)
    label = cand.label if cand is not None else CASE_LABEL
    saved = []

    flag_cols = [
        ("guard_fcap_clipped", "f_cap clipped to 1"),
        ("guard_capture_raw_exceeded_clipped_sum", "raw capture > clipped sum"),
        ("near_psi_d_08", "psi_d >= 0.8"),
        ("guard_psi_d_ge_1", "psi_d >= 1"),
        ("near_xi_d_08", "xi_d >= 0.8"),
        ("guard_xi_d_ge_1", "xi_d >= 1"),
        ("guard_lambda_near_clip", "lambda near xi clip"),
        ("guard_pressure_floor_b", "bulk p_eq floor"),
        ("guard_pressure_floor_d", "disl p_eq floor"),
        ("guard_nonfinite_R_or_N", "nonfinite R/N"),
        ("invalid_single_size_final", "invalid single-size flag"),
    ]

    for bu in BURNUPS:
        sub = gdf[np.isclose(gdf["burnup"], float(bu))].sort_values("T")
        if sub.empty:
            continue
        Ts = sub["T"].to_numpy()

        plt.figure(figsize=(11, 6.5))
        offset = 0
        for col, lab in flag_cols:
            y = sub[col].astype(float).to_numpy()
            # Offset each flag vertically so all activations are visible.
            plt.step(Ts, y + offset, where="mid", label=lab)
            offset += 1.25

        plt.yticks([])
        plt.xlabel("Temperature [K]")
        plt.title(f"{label}: numerical guard / invalidity indicators at {bu:.1f}% FIMA")
        plt.grid(True, axis="x", alpha=0.3)
        plt.legend(fontsize=8, loc="center left", bbox_to_anchor=(1.02, 0.5))
        saved.append(_save_guard_plot(f"{label}_guard_activation_flags_{bu:.1f}FIMA.png"))

    return saved

def plot_vacancy_absorption_summary(rows, cand=None):
    """For each burnup, plot final vacancy absorption diagnostics vs T."""
    gdf = guard_dataframe_from_rows(rows)
    label = cand.label if cand is not None else CASE_LABEL
    saved = []

    for bu in BURNUPS:
        sub = gdf[np.isclose(gdf["burnup"], float(bu))].sort_values("T")
        if sub.empty:
            continue
        Ts = sub["T"].to_numpy()

        fig, axes = plt.subplots(2, 2, figsize=(13, 8.5))

        ax = axes[0, 0]
        ax.semilogy(Ts, np.maximum(sub.get("Dv", pd.Series(np.nan, index=sub.index)).to_numpy(dtype=float), 1e-300), label=r"$D_v$")
        if "Dv1" in sub.columns:
            ax.semilogy(Ts, np.maximum(sub["Dv1"].to_numpy(dtype=float), 1e-300), "--", label=r"$D_{v,1}$")
        if "Dv2" in sub.columns:
            ax.semilogy(Ts, np.maximum(sub["Dv2"].to_numpy(dtype=float), 1e-300), "--", label=r"$D_{v,2}$")
        ax.set_ylabel(r"$D_v$ [m$^2$/s]")
        ax.set_title(f"{label}: vacancy diffusivity at {bu:.1f}% FIMA")
        ax.grid(True, which="both", alpha=0.3)
        ax.legend(fontsize=8)

        ax = axes[0, 1]
        if "dnvb_dt" in sub.columns:
            ax.semilogy(Ts, np.maximum(np.abs(sub["dnvb_dt"].to_numpy(dtype=float)), 1e-300), "--", label=r"$|dn_{v,b}/dt|$")
        if "dnvd_dt" in sub.columns:
            ax.semilogy(Ts, np.maximum(np.abs(sub["dnvd_dt"].to_numpy(dtype=float)), 1e-300), label=r"$|dn_{v,d}/dt|$")
        ax.set_ylabel(r"Vacancy absorption rate [vac m$^{-3}$ s$^{-1}$]")
        ax.set_title(f"{label}: final vacancy absorption rate at {bu:.1f}% FIMA")
        ax.grid(True, which="both", alpha=0.3)
        ax.legend(fontsize=8)

        ax = axes[1, 0]
        ax.semilogy(Ts, np.maximum(sub.get("p_b_over_eq", pd.Series(np.nan, index=sub.index)).to_numpy(dtype=float), 1e-300), "--", label=r"$p_b/p_{b,eq}$")
        ax.semilogy(Ts, np.maximum(sub.get("p_d_over_eq", pd.Series(np.nan, index=sub.index)).to_numpy(dtype=float), 1e-300), label=r"$p_d/p_{d,eq}$")
        ax.axhline(1.0, linestyle="--", linewidth=1)
        ax.set_xlabel("Temperature [K]")
        ax.set_ylabel("Pressure ratio [-]")
        ax.set_title(f"{label}: pressure driving force at {bu:.1f}% FIMA")
        ax.grid(True, which="both", alpha=0.3)
        ax.legend(fontsize=8)

        ax = axes[1, 1]
        ax.semilogy(Ts, np.maximum(sub.get("max_f_cap_step", pd.Series(np.nan, index=sub.index)).to_numpy(dtype=float), 1e-300), label="max f_cap step")
        ax.semilogy(Ts, np.maximum(sub.get("capture_raw_sum", pd.Series(np.nan, index=sub.index)).to_numpy(dtype=float), 1e-300), "--", label="sum raw capture hazard")
        ax.axhline(1.0, linestyle="--", linewidth=1)
        ax.set_xlabel("Temperature [K]")
        ax.set_ylabel("Capture diagnostic")
        ax.set_title(f"{label}: capture vs vacancy-growth consequences at {bu:.1f}% FIMA")
        ax.grid(True, which="both", alpha=0.3)
        ax.legend(fontsize=8)

        plt.tight_layout()
        saved.append(_save_guard_plot(f"{label}_vacancy_absorption_guard_summary_{bu:.1f}FIMA.png"))

    return saved

def make_guard_diagnostic_plots(rows, cand=None):
    """Create all final-state guard diagnostics."""
    if cand is None:
        cand = make_candidate()
    saved = []
    saved += plot_guard_geometry_summary(rows, cand)
    saved += plot_guard_activation_summary(rows, cand)
    saved += plot_vacancy_absorption_summary(rows, cand)

    gdf = guard_dataframe_from_rows(rows)
    out_csv = Path(OUTPUT_DIR) / f"{cand.label}_guard_diagnostics_final_state.csv"
    gdf.to_csv(out_csv, index=False)
    saved.append(out_csv)
    print("Guard diagnostic outputs:")
    for pth in saved:
        print(f"  - {pth}")
    return gdf, saved

def history_dataframe_for_point(T, burnup, cand=None):
    """Rerun one point with keep_history=True and build internal time-history diagnostics."""
    if cand is None:
        cand = make_candidate()
    out = run_model_point(float(T), float(burnup), cand, DT_H, N_MODES, keep_history=True)
    hist = out["hist"]
    hdf = pd.DataFrame({k: v for k, v in hist.items() if isinstance(v, list)})
    if hdf.empty:
        return hdf, out

    # Geometry from internal history.
    hdf["psi_b"] = [
        (R / wigner_seitz_delta(N) if _finite_positive(N) and math.isfinite(R) else math.nan)
        for R, N in zip(hdf["Rb"], hdf["Nb"])
    ]
    hdf["psi_d"] = [
        (R / wigner_seitz_delta(N) if _finite_positive(N) and math.isfinite(R) else math.nan)
        for R, N in zip(hdf["Rd"], hdf["Nd"])
    ]
    hdf["xi_b"] = hdf["swelling_b"]
    hdf["xi_d"] = hdf["swelling_d"]
    hdf["zeta_b"] = _zeta_from_psi_for_plot(hdf["psi_b"].to_numpy(dtype=float))
    hdf["zeta_d"] = _zeta_from_psi_for_plot(hdf["psi_d"].to_numpy(dtype=float))

    # Finite-difference vacancy absorption rates from history.
    time = hdf["time"].to_numpy(dtype=float)
    for key in ["nvb", "nvd"]:
        vals = hdf[key].to_numpy(dtype=float)
        rate = np.full_like(vals, np.nan, dtype=float)
        if len(vals) > 1:
            dt = np.diff(time)
            dv = np.diff(vals)
            ok = dt > 0
            rate[1:][ok] = dv[ok] / dt[ok]
        hdf[f"d{key}_dt_fd"] = rate

    # Inferred guard flags.
    hdf["guard_fcap_clipped"] = hdf["f_cap_step"] >= 1.0 - 1e-12
    hdf["guard_psi_b_ge_1"] = hdf["psi_b"] >= 1.0
    hdf["guard_psi_d_ge_1"] = hdf["psi_d"] >= 1.0
    hdf["near_psi_d_08"] = hdf["psi_d"] >= 0.8
    hdf["near_xi_d_08"] = hdf["xi_d"] >= 0.8
    hdf["guard_xi_d_ge_1"] = hdf["xi_d"] >= 1.0
    hdf["guard_pressure_floor_b"] = hdf["p_b_eq"] > 1e14
    hdf["guard_pressure_floor_d"] = hdf["p_d_eq"] > 1e14

    return hdf, out

def plot_guard_history(T, burnup, cand=None):
    """Detailed internal history plots for one temperature/burnup point."""
    if cand is None:
        cand = make_candidate()
    hdf, out = history_dataframe_for_point(T, burnup, cand)
    if hdf.empty:
        print("Empty history")
        return hdf, []

    label = cand.label
    suffix = f"{label}_history_guards_{float(burnup):.1f}FIMA_{float(T):.0f}K"
    saved = []
    x = hdf["burnup_percent_fima"].to_numpy(dtype=float)

    # Plot geometry and coalescence.
    fig, axes = plt.subplots(2, 2, figsize=(13, 8.5))
    ax = axes[0, 0]
    ax.plot(x, hdf["psi_b"], "--", label=r"$\psi_b$")
    ax.plot(x, hdf["psi_d"], label=r"$\psi_d$")
    ax.axhline(0.8, linestyle=":", linewidth=1)
    ax.axhline(1.0, linestyle="--", linewidth=1)
    ax.set_ylabel(r"$\psi$ [-]")
    ax.set_title(f"{label}: psi history, T={T:.0f} K, burnup={burnup:.1f}%")
    ax.grid(True, alpha=0.3)
    ax.legend(fontsize=8)

    ax = axes[0, 1]
    ax.plot(x, hdf["xi_b"], "--", label=r"$\xi_b$")
    ax.plot(x, hdf["xi_d"], label=r"$\xi_d$")
    ax.axhline(0.8, linestyle=":", linewidth=1)
    ax.axhline(1.0, linestyle="--", linewidth=1)
    ax.set_ylabel(r"$\xi=VN$ [-]")
    ax.set_title("Porosity / volume fraction")
    ax.grid(True, alpha=0.3)
    ax.legend(fontsize=8)

    ax = axes[1, 0]
    ax.semilogy(x, np.maximum(hdf["lambda_d"].to_numpy(dtype=float), 1e-300), label=r"$\lambda_d$")
    ax.axhline(_lambda_clip_value(), linestyle="--", linewidth=1, label=r"$\lambda(\xi=0.999999)$")
    ax.set_xlabel("Burnup [% FIMA]")
    ax.set_ylabel(r"$\lambda_d$ [-]")
    ax.grid(True, which="both", alpha=0.3)
    ax.legend(fontsize=8)

    ax = axes[1, 1]
    ax.semilogy(x, np.maximum(hdf["zeta_b"].to_numpy(dtype=float), 1e-300), "--", label=r"$\zeta_b$")
    ax.semilogy(x, np.maximum(hdf["zeta_d"].to_numpy(dtype=float), 1e-300), label=r"$\zeta_d$")
    ax.set_xlabel("Burnup [% FIMA]")
    ax.set_ylabel(r"$\zeta$ [-]")
    ax.grid(True, which="both", alpha=0.3)
    ax.legend(fontsize=8)

    plt.tight_layout()
    saved.append(_save_guard_plot(f"{suffix}_geometry.png"))

    # Plot capture and vacancy absorption.
    fig, axes = plt.subplots(2, 2, figsize=(13, 8.5))
    ax = axes[0, 0]
    ax.semilogy(x, np.maximum(hdf["f_cap_step"].to_numpy(dtype=float), 1e-300), label="f_cap step")
    ax.semilogy(x, np.maximum(hdf["cap_raw_step"].to_numpy(dtype=float), 1e-300), "--", label="cap raw step")
    ax.axhline(1.0, linestyle="--", linewidth=1)
    ax.set_ylabel("Capture")
    ax.set_title("Capture clipping history")
    ax.grid(True, which="both", alpha=0.3)
    ax.legend(fontsize=8)

    ax = axes[0, 1]
    ax.semilogy(x, np.maximum(np.abs(hdf["dnvb_dt_fd"].to_numpy(dtype=float)), 1e-300), "--", label=r"$|dn_{v,b}/dt|$ finite diff")
    ax.semilogy(x, np.maximum(np.abs(hdf["dnvd_dt_fd"].to_numpy(dtype=float)), 1e-300), label=r"$|dn_{v,d}/dt|$ finite diff")
    ax.set_ylabel(r"Vacancy absorption rate [vac m$^{-3}$ s$^{-1}$]")
    ax.set_title("Vacancy absorption history")
    ax.grid(True, which="both", alpha=0.3)
    ax.legend(fontsize=8)

    ax = axes[1, 0]
    pb_ratio = hdf["p_b"] / hdf["p_b_eq"].replace(0, np.nan)
    pd_ratio = hdf["p_d"] / hdf["p_d_eq"].replace(0, np.nan)
    ax.semilogy(x, np.maximum(pb_ratio.to_numpy(dtype=float), 1e-300), "--", label=r"$p_b/p_{b,eq}$")
    ax.semilogy(x, np.maximum(pd_ratio.to_numpy(dtype=float), 1e-300), label=r"$p_d/p_{d,eq}$")
    ax.axhline(1.0, linestyle="--", linewidth=1)
    ax.set_xlabel("Burnup [% FIMA]")
    ax.set_ylabel("Pressure ratio [-]")
    ax.grid(True, which="both", alpha=0.3)
    ax.legend(fontsize=8)

    ax = axes[1, 1]
    flag_cols = [
        ("guard_fcap_clipped", "f_cap clipped"),
        ("near_psi_d_08", "psi_d >= 0.8"),
        ("guard_psi_d_ge_1", "psi_d >= 1"),
        ("near_xi_d_08", "xi_d >= 0.8"),
        ("guard_xi_d_ge_1", "xi_d >= 1"),
        ("guard_pressure_floor_b", "bulk p_eq floor"),
    ]
    off = 0
    for col, lab in flag_cols:
        ax.step(x, hdf[col].astype(float) + off, where="post", label=lab)
        off += 1.25
    ax.set_yticks([])
    ax.set_xlabel("Burnup [% FIMA]")
    ax.set_title("Guard/invalidity flags")
    ax.grid(True, axis="x", alpha=0.3)
    ax.legend(fontsize=7, loc="center left", bbox_to_anchor=(1.02, 0.5))

    plt.tight_layout()
    saved.append(_save_guard_plot(f"{suffix}_capture_vacancy_flags.png"))

    # Save history CSV.
    csv_path = Path(OUTPUT_DIR) / f"{suffix}.csv"
    hdf.to_csv(csv_path, index=False)
    saved.append(csv_path)

    print(f"History guard outputs for T={T:g} K, burnup={burnup:g}% FIMA:")
    for pth in saved:
        print(f"  - {pth}")
    return hdf, saved


In [134]:

# ============================================================
# Run final-state guard diagnostics for the grid already computed by main()
# ============================================================

cand_diag = make_candidate()
guard_df, guard_saved_plots = make_guard_diagnostic_plots(rows, cand_diag)

# Show the most suspicious final grid points.
display_cols = [
    "burnup", "T",
    "max_f_cap_step", "capture_raw_sum", "capture_fraction_sum",
    "psi_d", "xi_d", "lambda_d",
    "p_d_over_eq", "p_b_over_eq",
    "guard_fcap_clipped", "near_psi_d_08", "guard_psi_d_ge_1",
    "near_xi_d_08", "guard_xi_d_ge_1",
    "guard_pressure_floor_b", "invalid_single_size_final",
]
display_cols = [c for c in display_cols if c in guard_df.columns]
suspect = guard_df[
    guard_df["guard_fcap_clipped"]
    | guard_df["near_psi_d_08"]
    | guard_df["near_xi_d_08"]
    | guard_df["guard_pressure_floor_b"]
    | guard_df["invalid_single_size_final"]
].sort_values(["burnup", "T"])

print("\nSuspicious final grid points:")
display(suspect[display_cols])


Guard diagnostic outputs:
  - 4test_UN_results/4test_UN_guard_geometry_1.1FIMA.png
  - 4test_UN_results/4test_UN_guard_geometry_1.3FIMA.png
  - 4test_UN_results/4test_UN_guard_geometry_3.2FIMA.png
  - 4test_UN_results/4test_UN_guard_activation_flags_1.1FIMA.png
  - 4test_UN_results/4test_UN_guard_activation_flags_1.3FIMA.png
  - 4test_UN_results/4test_UN_guard_activation_flags_3.2FIMA.png
  - 4test_UN_results/4test_UN_vacancy_absorption_guard_summary_1.1FIMA.png
  - 4test_UN_results/4test_UN_vacancy_absorption_guard_summary_1.3FIMA.png
  - 4test_UN_results/4test_UN_vacancy_absorption_guard_summary_3.2FIMA.png
  - 4test_UN_results/4test_UN_guard_diagnostics_final_state.csv

Suspicious final grid points:


,burnup,T,max_f_cap_step,capture_raw_sum,capture_fraction_sum,psi_d,xi_d,lambda_d,p_d_over_eq,p_b_over_eq,guard_fcap_clipped,near_psi_d_08,guard_psi_d_ge_1,near_xi_d_08,guard_xi_d_ge_1,guard_pressure_floor_b,invalid_single_size_final
44,1.3,1950.0,0.054508,1.412642,1.412642,0.764502,0.446823,4.511683,1.705596,1.000874,False,False,False,False,False,False,True
45,1.3,2000.0,0.090631,2.340120,2.340120,0.816816,0.544970,7.629858,2.989323,1.000618,False,True,False,False,False,False,True
63,3.2,1750.0,0.004038,1.227040,1.227040,0.742027,0.408563,3.842628,1.971962,1.000372,False,False,False,False,False,False,True
64,3.2,1800.0,0.009424,2.286566,2.286566,0.801592,0.515063,6.504977,4.155621,1.000565,False,True,False,False,False,False,True
65,3.2,1850.0,0.017642,3.354014,3.354014,0.832499,0.576967,9.391791,8.286176,1.000813,False,True,False,False,False,False,True
66,3.2,1900.0,0.031352,4.380878,4.380878,0.852081,0.618647,12.446016,15.400734,1.001037,False,True,False,False,False,False,True
67,3.2,1950.0,0.054508,5.349345,5.349345,0.866236,0.649992,15.734315,27.038219,1.001187,False,True,False,False,False,False,True
68,3.2,2000.0,0.090631,6.241663,6.241663,0.877469,0.675610,19.390342,45.492198,1.001275,False,True,False,False,False,False,True


In [135]:

# ============================================================
# Optional: detailed internal history for one suspicious point
# ============================================================
# Change these values and run this cell only when needed.
# It reruns ONE model point with keep_history=True, so it can take some time.

T_GUARD_HISTORY = 1650.0
BU_GUARD_HISTORY = 3.2

# Uncomment to run:
# guard_history_df, guard_history_saved = plot_guard_history(T_GUARD_HISTORY, BU_GUARD_HISTORY)
